In [4]:
import pandas as pd
import numpy as np
import re
import string
from pathlib import Path
import unicodedata
from time import perf_counter

In [5]:
BASE_DIR = Path("..")

TRAIN_DIR = BASE_DIR / "data" / "train"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

print(S1_PATH)
print(S2_PATH)
print(S3_PATH)
print(GT_PATH)

..\data\train\train_source1.tsv
..\data\train\train_source2.tsv
..\data\train\train_source3.tsv
..\data\train\train_ground_truth.tsv


In [6]:
s1_sample = pd.read_csv(
    S1_PATH,
    sep="\t",
    nrows=10
)

s2_sample = pd.read_csv(
    S2_PATH,
    sep="\t",
    nrows=10
)

s3_sample = pd.read_csv(
    S3_PATH,
    sep="\t",
    nrows=10
)

In [7]:
s1_sample

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US


In [8]:
s2_sample

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
5,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US
6,S2-584977605,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",India
7,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India
8,S2-721031885,Chavira Platinum Chimera LLC,"282 SAXONY DRIVE, FTT MITCHELL, KY",US
9,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",India


In [9]:
s3_sample

,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India
5,S3-578159284,LLC Hernandez Colonial Redwood,"2260- Housecreek Trail, Unit 407, Raleigh, Nor...",US
6,S3-121412624,Classic Equity Partners Group,"6885 Catalpa Bluff Ln, PO Box 5799, Dickinson,...",US
7,S3-249416830,Ectolumdrex dba X+ Madison Inc,"S03575 Cty Tk M, Town Of Buffalo, WI",US
8,S3-107644605,Animal Hanisch Hospirlg,"##8 Willow Oak Lane, Fl. 0, Saint Louis, Missouri",US
9,S3-160217003,Gomez Optimal,"343 Hempstead 161, Hope, Arkansas",US


In [10]:
print("S1 columns:")
print(s1_sample.columns.tolist())

print("\nS2 columns:")
print(s2_sample.columns.tolist())

print("\nS3 columns:")
print(s3_sample.columns.tolist())

S1 columns:
['entity_id', 'business_name', 'business_address', 'country']

S2 columns:
['entity_id', 'business_name', 'business_address', 'country']

S3 columns:
['entity_id', 'business_name', 'business_address', 'country']


In [11]:
s1_sample.iloc[0]

entity_id                                     S1-925783039
business_name                          Orelee's Barbershop
business_address    1795 Westchester Drive, High Point, NC
country                                                 US
Name: 0, dtype: object

In [12]:
SAMPLE_SIZE = 1000

s1_sample = pd.read_csv(
    S1_PATH,
    sep="\t",
    nrows=SAMPLE_SIZE
)

s2_sample = pd.read_csv(
    S2_PATH,
    sep="\t",
    nrows=SAMPLE_SIZE
)

s3_sample = pd.read_csv(
    S3_PATH,
    sep="\t",
    nrows=SAMPLE_SIZE
)

print("S1:", s1_sample.shape)
print("S2:", s2_sample.shape)
print("S3:", s3_sample.shape)

S1: (1000, 4)
S2: (1000, 4)
S3: (1000, 4)


In [13]:
for name, df in {
    "S1": s1_sample,
    "S2": s2_sample,
    "S3": s3_sample
}.items():

    print(f"\n{name}")
    print(df.isna().sum())


S1
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

S2
entity_id            0
business_name        0
business_address    35
country              0
dtype: int64

S3
entity_id            0
business_name        0
business_address    37
country              0
dtype: int64


In [14]:
def normalize_text(value):
    """
    Conservative normalization.

    - Handles missing values
    - Unicode normalization
    - Lowercase
    - Normalizes whitespace
    - Removes punctuation while preserving Unicode letters/numbers
    """
    if pd.isna(value):
        return ""

    value = str(value)

    # Normalize Unicode without deleting non-Latin scripts
    value = unicodedata.normalize("NFKC", value)

    # Lowercase
    value = value.lower()

    # Replace punctuation with spaces
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value).strip()

    return value

In [15]:
s1_sample["name_norm"] = s1_sample["business_name"].map(normalize_text)
s2_sample["name_norm"] = s2_sample["business_name"].map(normalize_text)
s3_sample["name_norm"] = s3_sample["business_name"].map(normalize_text)

In [16]:
s1_sample[["business_name", "name_norm"]].head(10)

,business_name,name_norm
0,Orelee's Barbershop,orelee s barbershop
1,Prime Money,prime money
2,B+ Retail Inc,b retail inc
3,Christ Chapel,christ chapel
4,Prabhav Business Center,prabhav business center
5,Custom Wealth Services LLC,custom wealth services llc
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited
7,Nexus Anchor Rain,nexus anchor rain
8,Moore Bitwise Inc,moore bitwise inc
9,Dermatology Green Medicine,dermatology green medicine


In [17]:
from collections import defaultdict

def build_name_index(df):
    index = defaultdict(list)

    for _, row in df.iterrows():
        name = row["name_norm"]

        if name:
            index[name].append(row["entity_id"])

    return index

In [18]:
s2_name_index = build_name_index(s2_sample)
s3_name_index = build_name_index(s3_sample)

print("S2 unique normalized names:", len(s2_name_index))
print("S3 unique normalized names:", len(s3_name_index))

S2 unique normalized names: 1000
S3 unique normalized names: 1000


In [19]:
def exact_name_candidates(s1_df, s2_index, s3_index):
    candidates = []

    for _, row in s1_df.iterrows():

        s1_id = row["entity_id"]
        name = row["name_norm"]

        if not name:
            continue

        # Search S2
        for candidate_id in s2_index.get(name, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": "exact_name"
            })

        # Search S3
        for candidate_id in s3_index.get(name, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": "exact_name"
            })

    return pd.DataFrame(candidates)

In [20]:
name_candidates = exact_name_candidates(
    s1_sample,
    s2_name_index,
    s3_name_index
)

name_candidates

,s1_entity_id,candidate_entity_id,source,block_type
0,S1-746273151,S3-538882911,S3,exact_name
1,S1-8092452,S3-551271002,S3,exact_name
2,S1-401074421,S3-303073926,S3,exact_name


In [21]:
pair = name_candidates.iloc[0]

s1_id = pair["s1_entity_id"]
candidate_id = pair["candidate_entity_id"]

print("S1 ID:", s1_id)
print("Candidate ID:", candidate_id)

S1 ID: S1-746273151
Candidate ID: S3-538882911


In [22]:
s1_match = s1_sample[
    s1_sample["entity_id"] == s1_id
]

s3_match = s3_sample[
    s3_sample["entity_id"] == candidate_id
]

display(
    s1_match[
        ["entity_id", "business_name", "business_address", "country", "name_norm"]
    ]
)

display(
    s3_match[
        ["entity_id", "business_name", "business_address", "country", "name_norm"]
    ]
)

,entity_id,business_name,business_address,country,name_norm
99,S1-746273151,Pediatric Dental Physicians Inc,"53 Laroche Lane, Hebron, ME",US,pediatric dental physicians inc


,entity_id,business_name,business_address,country,name_norm
646,S3-538882911,Pediatric Dental Physicians Inc.,NaN,US,pediatric dental physicians inc


In [23]:
print("Total candidate pairs:", len(name_candidates))

Total candidate pairs: 3


In [24]:
candidates_per_s1 = (
    name_candidates
    .groupby("s1_entity_id")
    .size()
)

print("S1 records with candidates:", len(candidates_per_s1))

if len(candidates_per_s1) > 0:
    print("Average:", candidates_per_s1.mean())
    print("Median:", candidates_per_s1.median())
    print("P95:", candidates_per_s1.quantile(0.95))
    print("P99:", candidates_per_s1.quantile(0.99))
    print("Maximum:", candidates_per_s1.max())

S1 records with candidates: 3
Average: 1.0
Median: 1.0
P95: 1.0
P99: 1.0
Maximum: 1


In [25]:
for _, pair in name_candidates.iterrows():
    print("=" * 70)
    print("S1:", pair["s1_entity_id"])
    print("Candidate:", pair["candidate_entity_id"])
    print("Source:", pair["source"])
    print("Block:", pair["block_type"])

    s1_row = s1_sample[
        s1_sample["entity_id"] == pair["s1_entity_id"]
    ]

    if pair["source"] == "S2":
        candidate_row = s2_sample[
            s2_sample["entity_id"] == pair["candidate_entity_id"]
        ]
    else:
        candidate_row = s3_sample[
            s3_sample["entity_id"] == pair["candidate_entity_id"]
        ]

    print("\nS1 record:")
    display(
        s1_row[
            ["entity_id", "business_name", "business_address", "country"]
        ]
    )

    print("\nCandidate record:")
    display(
        candidate_row[
            ["entity_id", "business_name", "business_address", "country"]
        ]
    )

S1: S1-746273151
Candidate: S3-538882911
Source: S3
Block: exact_name

S1 record:


,entity_id,business_name,business_address,country
99,S1-746273151,Pediatric Dental Physicians Inc,"53 Laroche Lane, Hebron, ME",US



Candidate record:


,entity_id,business_name,business_address,country
646,S3-538882911,Pediatric Dental Physicians Inc.,NaN,US


S1: S1-8092452
Candidate: S3-551271002
Source: S3
Block: exact_name

S1 record:


...

,entity_id,business_name,business_address,country
265,S3-551271002,Urban Nails,"Wilson Pike, Brentwood, Tennessee",US


S1: S1-401074421
Candidate: S3-303073926
Source: S3
Block: exact_name

S1 record:


,entity_id,business_name,business_address,country
968,S1-401074421,Chau Minerals LLC,"1620 Belmont Street, Unit B, Washington, DC",US



Candidate record:


,entity_id,business_name,business_address,country
895,S3-303073926,Chau Minerals LLC,"1620 Belmont St, Washingtont Ownship, District...",US


In [26]:
def tokenize(value):
    if not value:
        return []

    return value.split()

In [27]:
s1_sample["name_tokens"] = s1_sample["name_norm"].map(tokenize)
s2_sample["name_tokens"] = s2_sample["name_norm"].map(tokenize)
s3_sample["name_tokens"] = s3_sample["name_norm"].map(tokenize)

s1_sample[["business_name", "name_norm", "name_tokens"]].head(10)

,business_name,name_norm,name_tokens
0,Orelee's Barbershop,orelee s barbershop,"[orelee, s, barbershop]"
1,Prime Money,prime money,"[prime, money]"
2,B+ Retail Inc,b retail inc,"[b, retail, inc]"
3,Christ Chapel,christ chapel,"[christ, chapel]"
4,Prabhav Business Center,prabhav business center,"[prabhav, business, center]"
5,Custom Wealth Services LLC,custom wealth services llc,"[custom, wealth, services, llc]"
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"[consulting, nyasa, nursing, private, limited]"
7,Nexus Anchor Rain,nexus anchor rain,"[nexus, anchor, rain]"
8,Moore Bitwise Inc,moore bitwise inc,"[moore, bitwise, inc]"
9,Dermatology Green Medicine,dermatology green medicine,"[dermatology, green, medicine]"


In [28]:
def build_token_index(df):
    index = defaultdict(set)

    for _, row in df.iterrows():
        entity_id = row["entity_id"]

        for token in row["name_tokens"]:
            if token:
                index[token].add(entity_id)

    return index

s2_name_token_index = build_token_index(s2_sample)
s3_name_token_index = build_token_index(s3_sample)

print("Unique S2 name tokens:", len(s2_name_token_index))
print("Unique S3 name tokens:", len(s3_name_token_index))

Unique S2 name tokens: 1810
Unique S3 name tokens: 1760


In [29]:
for token in list(s2_name_token_index.keys())[:20]:
    print(token, "→", list(s2_name_token_index[token])[:5])

र → ['S2-920169761', 'S2-735885703', 'S2-214924024', 'S2-314146870', 'S2-615967906']
म → ['S2-920169761', 'S2-735885703', 'S2-214924024', 'S2-314146870', 'S2-615967906']
क → ['S2-343318003', 'S2-920169761', 'S2-166376419', 'S2-707472784', 'S2-913216366']
ट → ['S2-920169761', 'S2-735885703', 'S2-214924024', 'S2-314146870', 'S2-615967906']
ग → ['S2-343318003', 'S2-389587679', 'S2-631299191', 'S2-166376419', 'S2-314146870']
प → ['S2-920169761', 'S2-735885703', 'S2-214924024', 'S2-314146870', 'S2-615967906']
इव → ['S2-920169761', 'S2-735885703', 'S2-214924024', 'S2-314146870', 'S2-615967906']
ल → ['S2-920169761', 'S2-735885703', 'S2-913216366', 'S2-214924024', 'S2-314146870']
ड → ['S2-920169761', 'S2-735885703', 'S2-214924024', 'S2-314146870', 'S2-615967906']
holloway → ['S2-764573417']
peak → ['S2-888440125', 'S2-623626120', 'S2-764573417', 'S2-836840386', 'S2-162336156']
inc → ['S2-736654053', 'S2-635511596', 'S2-472544795', 'S2-143614907', 'S2-871017791']
seafood → ['S2-764573417']
आद →

In [30]:
def name_token_candidates(s1_df, s2_index, s3_index):
    candidates = []

    for _, row in s1_df.iterrows():

        s1_id = row["entity_id"]
        tokens = row["name_tokens"]

        matched_s2 = set()
        matched_s3 = set()

        for token in tokens:
            matched_s2.update(s2_index.get(token, set()))
            matched_s3.update(s3_index.get(token, set()))

        for candidate_id in matched_s2:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": "name_token"
            })

        for candidate_id in matched_s3:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": "name_token"
            })

    return pd.DataFrame(candidates)

In [31]:
token_candidates_df = name_token_candidates(
    s1_sample,
    s2_name_token_index,
    s3_name_token_index
)

print("Total token-block candidates:", len(token_candidates_df))

token_candidates_df.head(20)

Total token-block candidates: 155171


,s1_entity_id,candidate_entity_id,source,block_type
0,S1-925783039,S2-281482759,S2,name_token
1,S1-925783039,S2-402804051,S2,name_token
2,S1-925783039,S2-832217093,S2,name_token
3,S1-925783039,S2-222674377,S2,name_token
4,S1-925783039,S2-34975790,S2,name_token
5,S1-925783039,S2-531458823,S2,name_token
6,S1-925783039,S2-930194769,S2,name_token
7,S1-925783039,S2-209019917,S2,name_token
8,S1-925783039,S2-769808418,S2,name_token
9,S1-925783039,S2-64137184,S2,name_token


In [32]:
if len(token_candidates_df) > 0:

    token_counts = (
        token_candidates_df
        .groupby("s1_entity_id")
        .size()
    )

    print("Total candidate pairs:", len(token_candidates_df))
    print("S1 records with candidates:", len(token_counts))
    print("Average:", token_counts.mean())
    print("Median:", token_counts.median())
    print("P95:", token_counts.quantile(0.95))
    print("P99:", token_counts.quantile(0.99))
    print("Maximum:", token_counts.max())

else:
    print("No token candidates found.")

Total candidate pairs: 155171
S1 records with candidates: 982
Average: 158.0152749490835
Median: 176.0
P95: 341.0
P99: 359.18999999999994
Maximum: 387


In [33]:
token_frequency_s2 = {
    token: len(entity_ids)
    for token, entity_ids in s2_name_token_index.items()
}

token_frequency_s3 = {
    token: len(entity_ids)
    for token, entity_ids in s3_name_token_index.items()
}

In [34]:
print("\nMost common S3 tokens:")

for token, count in sorted(
    token_frequency_s3.items(),
    key=lambda x: x[1],
    reverse=True
)[:30]:
    print(repr(token), "→", count)


Most common S3 tokens:
'limited' → 132
'private' → 126
'llc' → 93
'inc' → 85
'ltd' → 72
'pvt' → 44
'com' → 43
'center' → 34
's' → 33
'partners' → 33
'services' → 30
'india' → 29
'ल' → 26
'ट' → 25
'l' → 25
'group' → 24
'c' → 24
'ड' → 23
'र' → 23
'म' → 23
'holdings' → 22
'co' → 20
'and' → 20
'care' → 19
'corp' → 19
'a' → 19
'lp' → 19
'प' → 19
'इव' → 17
'corporation' → 14


In [35]:
NAME_STOPWORDS = {
    "inc", "llc", "ltd", "pvt", "co", "corp", "corporation",
    "limited", "group", "company", "the", "and", "of",
}


def build_token_index(df, token_col, stopwords=None, max_doc_freq=None):
    """
    Build an inverted index: token -> list of entity_ids containing that token.

    stopwords: tokens to skip entirely (too generic to be useful).
    max_doc_freq: if set, any token appearing in MORE than this many records
                  is dropped from the index after counting (protects against
                  tokens we didn't think to add to the stopword list).
    """
    stopwords = stopwords or set()
    index = defaultdict(list)

    for _, row in df.iterrows():
        entity_id = row["entity_id"]
        tokens = row[token_col]

        for token in tokens:
            if len(token) < 2 or token in stopwords:
                continue
            index[token].append(entity_id)

    if max_doc_freq is not None:
        index = {
            token: ids
            for token, ids in index.items()
            if len(ids) <= max_doc_freq
        }

    return index


In [36]:
s2_name_token_index = build_token_index(s2_sample, "name_tokens", stopwords=NAME_STOPWORDS)
s3_name_token_index = build_token_index(s3_sample, "name_tokens", stopwords=NAME_STOPWORDS)

print("S2 name-token index size:", len(s2_name_token_index))
print("S3 name-token index size:", len(s3_name_token_index))

token_doc_freq = sorted(
    ((token, len(ids)) for token, ids in s2_name_token_index.items()),
    key=lambda x: -x[1],
)
print("\nMost frequent S2 name tokens:")
for token, freq in token_doc_freq[:15]:
    print(f"  {token!r}: {freq}")


S2 name-token index size: 1639
S3 name-token index size: 1603

Most frequent S2 name tokens:
  'private': 121
  'com': 47
  'center': 38
  'partners': 35
  'इव': 32
  'services': 25
  'care': 25
  'india': 25
  'holdings': 18
  'llp': 17
  'associates': 14
  'global': 13
  'industries': 13
  'brothers': 10
  'service': 10


In [37]:
def token_block_candidates(s1_df, token_col, s2_index, s3_index, block_type):

    candidates = []

    for _, row in s1_df.iterrows():
        s1_id = row["entity_id"]
        tokens = row[token_col]

        if not tokens:
            continue

        matched_s2 = set()
        matched_s3 = set()

        for token in tokens:
            matched_s2.update(s2_index.get(token, []))
            matched_s3.update(s3_index.get(token, []))

        for candidate_id in matched_s2:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": block_type,
            })

        for candidate_id in matched_s3:
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": block_type,
            })

    return pd.DataFrame(candidates)


In [38]:
name_token_candidates = token_block_candidates(
    s1_sample,
    "name_tokens",
    s2_name_token_index,
    s3_name_token_index,
    block_type="name_token",
)

print("Name-token candidate pairs:", len(name_token_candidates))
print("Exact-name candidate pairs (Strategy 1):", len(name_candidates))
name_token_candidates.head(10)


Name-token candidate pairs: 60110
Exact-name candidate pairs (Strategy 1): 3


,s1_entity_id,candidate_entity_id,source,block_type
0,S1-925783039,S2-415733261,S2,name_token
1,S1-773889195,S2-519912199,S2,name_token
2,S1-773889195,S3-325154278,S3,name_token
3,S1-773889195,S3-941511369,S3,name_token
4,S1-377745466,S2-325615566,S2,name_token
5,S1-377745466,S2-202057528,S2,name_token
6,S1-377745466,S2-477675434,S2,name_token
7,S1-377745466,S3-869462086,S3,name_token
8,S1-377745466,S3-657028395,S3,name_token
9,S1-133037285,S2-777835859,S2,name_token


In [39]:
def candidate_stats(candidates_df, label=""):

    if label:
        print(f"--- {label} ---")

    if len(candidates_df) == 0:
        print("No candidates generated.")
        return None

    per_s1 = candidates_df.groupby("s1_entity_id").size()

    print("S1 records with at least one candidate:", len(per_s1))
    print("Average candidates per S1:", per_s1.mean())
    print("Median candidates per S1:", per_s1.median())
    print("P95:", per_s1.quantile(0.95))
    print("P99:", per_s1.quantile(0.99))
    print("Max:", per_s1.max())

    return per_s1


_ = candidate_stats(name_token_candidates, label="Name-token block (sample)")


--- Name-token block (sample) ---
S1 records with at least one candidate: 888
Average candidates per S1: 67.69144144144144
Median candidates per S1: 13.0
P95: 256.0
P99: 280.0
Max: 301


In [40]:
def normalize_address(value):

    return normalize_text(value)


s1_sample["addr_norm"] = s1_sample["business_address"].map(normalize_address)
s2_sample["addr_norm"] = s2_sample["business_address"].map(normalize_address)
s3_sample["addr_norm"] = s3_sample["business_address"].map(normalize_address)

s1_sample[["business_address", "addr_norm"]].head(10)


,business_address,addr_norm
0,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


In [41]:
missing_addr_rows = s3_sample[s3_sample["business_address"].isna()]
missing_addr_rows[["entity_id", "business_name", "business_address", "addr_norm"]]

,entity_id,business_name,business_address,addr_norm
1,S3-859268022,International South Consultants Private Ltd,NaN,
52,S3-92001570,Davis Acquisition,NaN,
105,S3-299576808,"Morales, Godina & Pánkow Associates LLC",NaN,
110,S3-924073964,The Nancy Nancy Bunten Energy,NaN,
125,S3-588120445,Anand Agro Private Enterprises,NaN,
189,S3-32634960,Shyam Foundation Private Limited,NaN,
190,S3-509442398,New Delhi Techn0logies Private Limited,NaN,
231,S3-987949855,Southern பவர்,NaN,
251,S3-301461097,Al Macrktig Pvt Ltd,NaN,
302,S3-180062010,Beus Inc. Center,NaN,


In [42]:
s1_sample["addr_tokens"] = s1_sample["addr_norm"].map(tokenize)
s2_sample["addr_tokens"] = s2_sample["addr_norm"].map(tokenize)
s3_sample["addr_tokens"] = s3_sample["addr_norm"].map(tokenize)

s1_sample[["business_address", "addr_norm", "addr_tokens"]].head(10)


,business_address,addr_norm,addr_tokens
0,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,"[1795, westchester, drive, high, point, nc]"
1,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,"[17560, ellis, road, tahlequah, ok]"
2,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,"[1712, montebello, avenue, phoenix, az]"
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,"[2100, cameron, drive, unit, apartment, g, dun..."
4,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,"[797, lake, town, block, a, kolkata, howrah, w..."
5,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,"[oh, columbus, 5559, orville, avenue]"
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,"[2505, tower, 1, oakwood, runwal, greens, mulu..."
7,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,"[1111, church, street, unit, 2007, nashville, tn]"
8,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,"[337, oakland, avenue, michigan, city, in]"
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,"[294, meadowcreek, drive, unit, unit, 2, villa..."


In [43]:
ADDRESS_STOPWORDS = {
    "street", "st", "road", "rd", "avenue", "ave", "lane", "ln",
    "drive", "dr", "near", "opp", "no", "nagar",
    "north", "south", "east", "west",
}

s2_addr_token_index = build_token_index(s2_sample, "addr_tokens", stopwords=ADDRESS_STOPWORDS)
s3_addr_token_index = build_token_index(s3_sample, "addr_tokens", stopwords=ADDRESS_STOPWORDS)

print("S2 address-token index size:", len(s2_addr_token_index))
print("S3 address-token index size:", len(s3_addr_token_index))

S2 address-token index size: 3147
S3 address-token index size: 2989


In [44]:
address_token_candidates = token_block_candidates(
    s1_sample,
    "addr_tokens",
    s2_addr_token_index,
    s3_addr_token_index,
    block_type="address_token",
)

print("Address-token candidate pairs:", len(address_token_candidates))
_ = candidate_stats(address_token_candidates, label="Address-token block (sample)")


Address-token candidate pairs: 85602
--- Address-token block (sample) ---
S1 records with at least one candidate: 1000
Average candidates per S1: 85.602
Median candidates per S1: 60.0
P95: 228.0
P99: 302.04999999999995
Max: 408


In [45]:
import re as _re  


def extract_leading_number(value):

    if not value:
        return None

    match = re.search(r"\d+", value)
    return match.group(0) if match else None


s1_sample["addr_number"] = s1_sample["addr_norm"].map(extract_leading_number)
s2_sample["addr_number"] = s2_sample["addr_norm"].map(extract_leading_number)
s3_sample["addr_number"] = s3_sample["addr_norm"].map(extract_leading_number)

s1_sample[["business_address", "addr_norm", "addr_number"]].head(10)

,business_address,addr_norm,addr_number
0,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,1795
1,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,17560
2,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,1712
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,2100
4,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,797
5,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,5559
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,2505
7,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,1111
8,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,337
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,294


In [46]:
def build_number_index(df, number_col, max_doc_freq=None):

    index = defaultdict(list)

    for _, row in df.iterrows():
        number = row[number_col]
        if number:
            index[number].append(row["entity_id"])

    if max_doc_freq is not None:
        index = {
            number: ids
            for number, ids in index.items()
            if len(ids) <= max_doc_freq
        }

    return index

MAX_NUMBER_DOC_FREQ = 5

s2_number_index = build_number_index(s2_sample, "addr_number", max_doc_freq=MAX_NUMBER_DOC_FREQ)
s3_number_index = build_number_index(s3_sample, "addr_number", max_doc_freq=MAX_NUMBER_DOC_FREQ)

print("S2 address-number index size:", len(s2_number_index))
print("S3 address-number index size:", len(s3_number_index))


S2 address-number index size: 631
S3 address-number index size: 609


In [47]:
def address_number_candidates(s1_df, s2_index, s3_index):
   
    candidates = []

    for _, row in s1_df.iterrows():
        s1_id = row["entity_id"]
        number = row["addr_number"]

        if not number:
            continue

        for candidate_id in s2_index.get(number, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S2",
                "block_type": "address_number",
            })

        for candidate_id in s3_index.get(number, []):
            candidates.append({
                "s1_entity_id": s1_id,
                "candidate_entity_id": candidate_id,
                "source": "S3",
                "block_type": "address_number",
            })

    return pd.DataFrame(candidates)


number_candidates = address_number_candidates(s1_sample, s2_number_index, s3_number_index)

print("Address-number candidate pairs:", len(number_candidates))
_ = candidate_stats(number_candidates, label="Address-number block (sample)")


Address-number candidate pairs: 1074
--- Address-number block (sample) ---
S1 records with at least one candidate: 393
Average candidates per S1: 2.732824427480916
Median candidates per S1: 2.0
P95: 6.0
P99: 7.079999999999984
Max: 8


In [48]:
def combine_candidate_blocks(block_dfs):
   
    non_empty = [df for df in block_dfs if len(df) > 0]

    if not non_empty:
        return pd.DataFrame(
            columns=["s1_entity_id", "candidate_entity_id", "source", "retrieved_by"]
        )

    combined = pd.concat(non_empty, ignore_index=True)

    combined = (
        combined
        .groupby(["s1_entity_id", "candidate_entity_id", "source"])["block_type"]
        .agg(lambda block_types: sorted(set(block_types)))
        .reset_index()
        .rename(columns={"block_type": "retrieved_by"})
    )

    return combined


combined_candidates = combine_candidate_blocks([
    name_candidates,
    name_token_candidates,
    address_token_candidates,
    number_candidates,
])

total_raw = (
    len(name_candidates)
    + len(name_token_candidates)
    + len(address_token_candidates)
    + len(number_candidates)
)

print("Total candidate rows before dedup (sum across blocks):", total_raw)
print("Total unique candidate pairs after union + dedup:", len(combined_candidates))
combined_candidates.head(10)


Total candidate rows before dedup (sum across blocks): 146789
Total unique candidate pairs after union + dedup: 139156


,s1_entity_id,candidate_entity_id,source,retrieved_by
0,S1-100146655,S2-146516261,S2,[address_token]
1,S1-100146655,S2-147317655,S2,[address_token]
2,S1-100146655,S2-169701381,S2,[address_token]
3,S1-100146655,S2-172058052,S2,[address_token]
4,S1-100146655,S2-194307610,S2,[address_token]
5,S1-100146655,S2-200440168,S2,[address_token]
6,S1-100146655,S2-215156674,S2,[address_token]
7,S1-100146655,S2-259858270,S2,[address_token]
8,S1-100146655,S2-265287731,S2,[address_token]
9,S1-100146655,S2-278217555,S2,[address_token]


In [49]:
from collections import Counter

block_membership_counts = Counter()
for retrieved_by in combined_candidates["retrieved_by"]:
    for block_type in retrieved_by:
        block_membership_counts[block_type] += 1

print("How many combined candidate pairs each block contributed to:")
for block_type, count in block_membership_counts.most_common():
    print(f"  {block_type}: {count}")

multi_block_count = (combined_candidates["retrieved_by"].map(len) > 1).sum()
print(f"\nPairs retrieved by more than one block: {multi_block_count} / {len(combined_candidates)}")

combined_candidates[combined_candidates["retrieved_by"].map(len) > 1].head(10)

How many combined candidate pairs each block contributed to:
  address_token: 85602
  name_token: 60110
  address_number: 1074
  exact_name: 3

Pairs retrieved by more than one block: 7570 / 139156


,s1_entity_id,candidate_entity_id,source,retrieved_by
228,S1-102971431,S2-155636061,S2,"[address_token, name_token]"
230,S1-102971431,S2-164651728,S2,"[address_token, name_token]"
234,S1-102971431,S2-178384635,S2,"[address_token, name_token]"
291,S1-102971431,S2-54720637,S2,"[address_number, address_token]"
294,S1-102971431,S2-556747559,S2,"[address_token, name_token]"
299,S1-102971431,S2-577969603,S2,"[address_token, name_token]"
309,S1-102971431,S2-652381513,S2,"[address_token, name_token]"
310,S1-102971431,S2-663049678,S2,"[address_token, name_token]"
320,S1-102971431,S2-708389947,S2,"[address_token, name_token]"
329,S1-102971431,S2-757169960,S2,"[address_token, name_token]"


In [50]:
_ = candidate_stats(combined_candidates, label="Combined blocks (sample, all 4 strategies)")

s1_with_candidates = set(combined_candidates["s1_entity_id"])
s1_without_candidates = set(s1_sample["entity_id"]) - s1_with_candidates

print(f"\nS1 records with at least one candidate: {len(s1_with_candidates)} / {len(s1_sample)}")
print(f"S1 records with NO candidates from any block: {len(s1_without_candidates)}")


--- Combined blocks (sample, all 4 strategies) ---
S1 records with at least one candidate: 1000
Average candidates per S1: 139.156
Median candidates per S1: 84.5
P95: 396.04999999999995
P99: 475.06999999999994
Max: 573

S1 records with at least one candidate: 1000 / 1000
S1 records with NO candidates from any block: 0


In [51]:
import json as _json

VALIDATION_DIR = BASE_DIR / "code" / "business_entity_resolution" / "src" / "validation_artifacts"

VAL_S1_IDS_PATH = VALIDATION_DIR / "val_s1_ids.txt"
VAL_GT_DICT_PATH = VALIDATION_DIR / "val_gt_dict.json"

print(VAL_S1_IDS_PATH)
print(VAL_GT_DICT_PATH)


..\code\business_entity_resolution\src\validation_artifacts\val_s1_ids.txt
..\code\business_entity_resolution\src\validation_artifacts\val_gt_dict.json


In [52]:
def load_val_s1_ids(path):
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def load_val_gt_dict(path):

    raw = path.read_text(encoding="utf-8", errors="replace")
    entries = re.findall(r'"(S1-[^"]+)":\s*\[(.*?)\]', raw, flags=re.DOTALL)

    val_gt = {}
    corrupted = []

    for s1_id, body in entries:
        raw_ids = re.findall(r'"([^"]*)"', body)
        cleaned = {i for i in raw_ids if i not in ("nan", "")}

        bad = [i for i in cleaned if not i.startswith(("S2-", "S3-"))]
        if bad:
            corrupted.append((s1_id, bad))

        val_gt[s1_id] = {i for i in cleaned if i.startswith(("S2-", "S3-"))}

    return val_gt, corrupted


val_s1_ids = load_val_s1_ids(VAL_S1_IDS_PATH)
val_gt_dict, corrupted_entries = load_val_gt_dict(VAL_GT_DICT_PATH)

print("Validation S1 IDs:", len(val_s1_ids))
print("Ground-truth entries loaded:", len(val_gt_dict))
print("Corrupted entries found:", len(corrupted_entries))

missing_from_dict = [s1_id for s1_id in val_s1_ids if s1_id not in val_gt_dict]
print(f"Validation IDs with NO ground-truth entry at all: {len(missing_from_dict)}")
if missing_from_dict:
    print("  (first 5):", missing_from_dict[:5])


Validation S1 IDs: 415947
Ground-truth entries loaded: 441289
Corrupted entries found: 1
Validation IDs with NO ground-truth entry at all: 72
  (first 5): ['S1-107932707', 'S1-12116720', 'S1-143649009', 'S1-159064592', 'S1-1622681']


In [53]:
def compute_candidate_recall(candidates_df, val_ids, val_gt):

    candidates_by_s1 = (
        candidates_df.groupby("s1_entity_id")["candidate_entity_id"]
        .apply(set)
        .to_dict()
    )

    per_entity_recall = []
    skipped = []

    for s1_id in val_ids:
        if s1_id not in val_gt:
            skipped.append(s1_id)
            continue

        true_matches = val_gt[s1_id]
        candidate_ids = candidates_by_s1.get(s1_id, set())

        if len(true_matches) == 0:
            per_entity_recall.append(1.0)
        else:
            found = len(true_matches & candidate_ids)
            per_entity_recall.append(found / len(true_matches))

    macro_recall = sum(per_entity_recall) / len(per_entity_recall) if per_entity_recall else 0.0

    return {
        "macro_candidate_recall": macro_recall,
        "n_evaluated": len(per_entity_recall),
        "n_skipped_no_ground_truth": len(skipped),
    }

print("compute_candidate_recall() is defined and ready for the full-scale run.")


compute_candidate_recall() is defined and ready for the full-scale run.


In [54]:
val_ids = val_s1_ids
val_gt = val_gt_dict

print("Validation S1 IDs:", len(val_ids))
print("Ground-truth entries:", len(val_gt))
print("Corrupted entries:", len(corrupted_entries))

Validation S1 IDs: 415947
Ground-truth entries: 441289
Corrupted entries: 1


In [55]:
print("First 5 validation S1 IDs:")
print(val_ids[:5])

print("\nFirst 5 ground-truth entries:")
for s1_id in list(val_gt.keys())[:5]:
    print(s1_id, "→", val_gt[s1_id])

First 5 validation S1 IDs:
['S1-100001512', 'S1-100002771', 'S1-100004023', 'S1-100005592', 'S1-100005874']

First 5 ground-truth entries:
S1-55344266 → {'S2-197070651', 'S3-384364074', 'S3-478195123', 'S2-249013014'}
S1-7293388 → {'S2-157073701', 'S2-442723188', 'S3-523120965', 'S2-7028416'}
S1-546142636 → {'S2-392804085', 'S2-487600131', 'S2-582477216', 'S3-200008747', 'S3-249331830', 'S3-729771680'}
S1-727602285 → {'S2-976870196', 'S2-947230367', 'S2-138660620', 'S2-23141904', 'S3-204655096'}
S1-840162906 → {'S3-799520471', 'S3-762681944', 'S3-800978181', 'S3-139858754', 'S2-129529678'}


In [56]:
VALIDATION_S1_IDS = set(val_ids)

s1_val = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s1_val = s1_val[
    s1_val["entity_id"].isin(VALIDATION_S1_IDS)
].copy()

print("Validation S1 rows loaded:", len(s1_val))

Validation S1 rows loaded: 415946


In [57]:
s1_val["name_norm"] = (
    s1_val["business_name"]
    .map(normalize_text)
)

print(
    "Validation S1 records with non-empty names:",
    (s1_val["name_norm"] != "").sum()
)

Validation S1 records with non-empty names: 415946


In [58]:
def build_full_name_index(path, chunk_size=200_000):

    index = defaultdict(set)

    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name"],
        chunksize=chunk_size
    ):

        total_rows += len(chunk)

        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_text)
        )

        chunk = chunk[chunk["name_norm"] != ""]

        for name, group in chunk.groupby("name_norm"):

            index[name].update(
                group["entity_id"].tolist()
            )

        print(
            f"Processed {total_rows:,} rows..."
        )

    return index

In [59]:
start = perf_counter()

s2_full_name_index = build_full_name_index(
    S2_PATH,
    chunk_size=200_000
)

elapsed = perf_counter() - start

print("\nS2 index built.")
print("Unique normalized names:", len(s2_full_name_index))
print(f"Time: {elapsed:.2f} seconds")

Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,034,616 rows...

S2 index built.
Unique normalized names: 4025056
Time: 219.02 seconds


In [60]:
start = perf_counter()

s3_full_name_index = build_full_name_index(
    S3_PATH,
    chunk_size=200_000
)

elapsed = perf_counter() - start

print("\nS3 index built.")
print("Unique normalized names:", len(s3_full_name_index))
print(f"Time: {elapsed:.2f} seconds")

Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,200,000 rows...
Processed 5,285,603 rows...

S3 index built.
Unique normalized names: 4280399
Time: 505.25 seconds


In [61]:
validation_name_candidates = exact_name_candidates(
    s1_val,
    s2_full_name_index,
    s3_full_name_index
)

print(
    "Total exact-name candidate pairs:",
    len(validation_name_candidates)
)

Total exact-name candidate pairs: 4124695


In [62]:
_ = candidate_stats(
    validation_name_candidates,
    label="Exact normalized-name block (validation)"
)

--- Exact normalized-name block (validation) ---
S1 records with at least one candidate: 295286
Average candidates per S1: 13.96847463137433
Median candidates per S1: 2.0
P95: 78.0
P99: 200.0
Max: 459


In [114]:
exact_name_recall = compute_candidate_recall(
    validation_name_candidates,
    val_ids,
    val_gt
)

print(exact_name_recall)

{'macro_candidate_recall': 0.2627617175664065, 'n_evaluated': 415875, 'n_skipped_no_ground_truth': 72}


Exact Normalized Name Blocking

Validation results:

- Candidate recall: 26.28%
- Total candidate pairs: 4,124,695
- Average candidates per S1: 13.97
- Median: 2
- P95: 78
- P99: 200
- Maximum: 459

In [67]:
from collections import Counter

def token_frequency_full(path, chunk_size=200_000):
    frequency = Counter()
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["business_name"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        for name in chunk["business_name"]:
            normalized = normalize_text(name)

            if not normalized:
                continue

            tokens = set(normalized.split())
            frequency.update(tokens)

        print(f"Processed {total_rows:,} rows...")

    return frequency

In [68]:
start = perf_counter()

s2_token_frequency = token_frequency_full(S2_PATH)

print("\nS2 token frequency built.")
print("Unique tokens:", len(s2_token_frequency))
print(f"Time: {perf_counter() - start:.2f} seconds")

Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,034,616 rows...

S2 token frequency built.
Unique tokens: 810514
Time: 49.67 seconds


In [69]:
start = perf_counter()

s3_token_frequency = token_frequency_full(S3_PATH)

print("\nS3 token frequency built.")
print("Unique tokens:", len(s3_token_frequency))
print(f"Time: {perf_counter() - start:.2f} seconds")

Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,200,000 rows...
Processed 5,285,603 rows...

S3 token frequency built.
Unique tokens: 863014
Time: 49.40 seconds


In [70]:
print("TOP S2 TOKENS")
for token, count in s2_token_frequency.most_common(30):
    print(repr(token), "→", count)

print("\nTOP S3 TOKENS")
for token, count in s3_token_frequency.most_common(30):
    print(repr(token), "→", count)

TOP S2 TOKENS
'limited' → 528795
'llc' → 528303
'private' → 528131
'inc' → 400229
'ltd' → 398464
'ल' → 237102
'ट' → 232862
'र' → 226825
'ड' → 208515
'प' → 206868
'म' → 206488
'com' → 201464
'center' → 193696
'pvt' → 189270
'स' → 160325
'इव' → 157955
'partners' → 152694
'services' → 145239
'corp' → 141773
'co' → 139561
'group' → 139390
's' → 138303
'and' → 118201
'c' → 113281
'india' → 110684
'क' → 105891
'holdings' → 97019
'l' → 93771
'care' → 88788
'of' → 87992

TOP S3 TOKENS
'limited' → 673480
'private' → 639861
'llc' → 571362
'ltd' → 427781
'inc' → 419917
'center' → 220809
'pvt' → 220353
'com' → 211116
'services' → 166621
'partners' → 161955
's' → 147647
'group' → 144614
'co' → 143838
'corp' → 140236
'ल' → 134117
'ट' → 132429
'र' → 128844
'and' → 122220
'ड' → 117653
'प' → 116694
'म' → 116370
'c' → 113132
'india' → 112959
'holdings' → 99024
'care' → 95821
'of' → 90980
'स' → 90553
'l' → 89809
'इव' → 88517
'a' → 79896


In [71]:
s2_token_frequency
s3_token_frequency

Counter({'limited': 673480,
         'private': 639861,
         'llc': 571362,
         'ltd': 427781,
         'inc': 419917,
         'center': 220809,
         'pvt': 220353,
         'com': 211116,
         'services': 166621,
         'partners': 161955,
         's': 147647,
         'group': 144614,
         'co': 143838,
         'corp': 140236,
         'ल': 134117,
         'ट': 132429,
         'र': 128844,
         'and': 122220,
         'ड': 117653,
         'प': 116694,
         'म': 116370,
         'c': 113132,
         'india': 112959,
         'holdings': 99024,
         'care': 95821,
         'of': 90980,
         'स': 90553,
         'l': 89809,
         'इव': 88517,
         'a': 79896,
         'llp': 77190,
         'associates': 76679,
         'd': 64840,
         'service': 63953,
         'क': 59798,
         'p': 58730,
         'corporation': 58097,
         'enterprises': 56858,
         'the': 54457,
         'health': 53452,
         'industries': 534

In [72]:
thresholds = [100, 500, 1000, 5000, 10000, 25000, 50000]

print("S2 tokens surviving each threshold")
print("-" * 50)

for threshold in thresholds:
    count = sum(
        freq <= threshold
        for freq in s2_token_frequency.values()
    )
    print(f"<= {threshold:>6}: {count:,} tokens")

print("\nS3 tokens surviving each threshold")
print("-" * 50)

for threshold in thresholds:
    count = sum(
        freq <= threshold
        for freq in s3_token_frequency.values()
    )
    print(f"<= {threshold:>6}: {count:,} tokens")

S2 tokens surviving each threshold
--------------------------------------------------
<=    100: 801,413 tokens
<=    500: 807,680 tokens
<=   1000: 808,972 tokens
<=   5000: 809,965 tokens
<=  10000: 810,176 tokens
<=  25000: 810,411 tokens
<=  50000: 810,474 tokens

S3 tokens surviving each threshold
--------------------------------------------------
<=    100: 853,517 tokens
<=    500: 860,216 tokens
<=   1000: 861,464 tokens
<=   5000: 862,459 tokens
<=  10000: 862,668 tokens
<=  25000: 862,936 tokens
<=  50000: 862,972 tokens


In [73]:
def useful_tokens(name, frequency, threshold):
    normalized = normalize_text(name)

    if not normalized:
        return []

    return [
        token
        for token in set(normalized.split())
        if frequency.get(token, 0) <= threshold
    ]

In [74]:
for threshold in thresholds:
    usable = 0
    total_tokens = 0

    for name in s1_val["business_name"]:
        tokens = useful_tokens(
            name,
            s2_token_frequency,
            threshold
        )

        if tokens:
            usable += 1
            total_tokens += len(tokens)

    print(
        f"Threshold {threshold:>6}: "
        f"{usable:,} / {len(s1_val):,} S1 records have "
        f"at least one usable token | "
        f"avg usable tokens = "
        f"{total_tokens / len(s1_val):.2f}"
    )

Threshold    100: 136,523 / 415,946 S1 records have at least one usable token | avg usable tokens = 0.37
Threshold    500: 199,434 / 415,946 S1 records have at least one usable token | avg usable tokens = 0.59
Threshold   1000: 241,727 / 415,946 S1 records have at least one usable token | avg usable tokens = 0.82
Threshold   5000: 335,261 / 415,946 S1 records have at least one usable token | avg usable tokens = 1.27
Threshold  10000: 367,933 / 415,946 S1 records have at least one usable token | avg usable tokens = 1.52
Threshold  25000: 414,286 / 415,946 S1 records have at least one usable token | avg usable tokens = 2.24
Threshold  50000: 415,910 / 415,946 S1 records have at least one usable token | avg usable tokens = 2.40


In [75]:
TOKEN_FREQ_THRESHOLD = 25_000

def build_filtered_token_index(
    path,
    token_frequency,
    max_frequency=25_000,
    chunk_size=200_000
):
    index = defaultdict(set)

    total_rows = 0
    kept_tokens = 0

    allowed_tokens = {
        token
        for token, freq in token_frequency.items()
        if freq <= max_frequency
    }

    print("Allowed tokens:", len(allowed_tokens))

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        for entity_id, name in zip(
            chunk["entity_id"],
            chunk["business_name"]
        ):
            normalized = normalize_text(name)

            if not normalized:
                continue

            tokens = set(normalized.split())

            for token in tokens:
                if token in allowed_tokens:
                    index[token].add(entity_id)

        print(f"Processed {total_rows:,} rows...")

    print(
        f"Finished. Index contains {len(index):,} tokens."
    )

    return index


In [76]:
start = perf_counter()

s2_filtered_token_index = build_filtered_token_index(
    S2_PATH,
    s2_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS2 filtered token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Allowed tokens: 810411
Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,034,616 rows...
Finished. Index contains 810,411 tokens.

S2 filtered token index built in 75.40 seconds


In [77]:
start = perf_counter()

s3_filtered_token_index = build_filtered_token_index(
    S3_PATH,
    s3_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS3 filtered token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Allowed tokens: 862936
Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,200,000 rows...
Processed 5,285,603 rows...
Finished. Index contains 862,936 tokens.

S3 filtered token index built in 54.72 seconds


In [78]:
def evaluate_filtered_token_block(
    s1_df,
    s2_index,
    s3_index,
    val_gt,
    max_s1=None
):

    total_candidates = 0
    s1_with_candidates = 0

    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if max_s1 is not None and i >= max_s1:
            break

        s1_id = row["entity_id"]

        normalized = normalize_text(row["business_name"])

        if not normalized:
            candidate_ids = set()
        else:
            tokens = set(normalized.split())

            candidate_ids = set()

            for token in tokens:
                candidate_ids.update(
                    s2_index.get(token, set())
                )
                candidate_ids.update(
                    s3_index.get(token, set())
                )

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates

        if n_candidates > 0:
            s1_with_candidates += 1

        per_s1_counts.append(n_candidates)

        # Recall
        if s1_id not in val_gt:
            skipped += 1
        else:
            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:
                recall_sum += 1.0
            else:
                found = len(
                    true_matches & candidate_ids
                )

                recall_sum += (
                    found / len(true_matches)
                )

            evaluated += 1

        if (i + 1) % 10_000 == 0:
            print(
                f"Processed {i + 1:,} S1 records | "
                f"candidates so far: {total_candidates:,}"
            )

    import numpy as np

    counts = np.array(per_s1_counts)

    result = {
        "total_candidate_pairs": total_candidates,
        "s1_records_with_candidates": s1_with_candidates,
        "average_candidates_per_s1": counts.mean(),
        "median_candidates_per_s1": np.median(counts),
        "p95_candidates_per_s1": np.quantile(counts, 0.95),
        "p99_candidates_per_s1": np.quantile(counts, 0.99),
        "max_candidates_per_s1": counts.max(),
        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated > 0 else 0.0
        ),
        "n_evaluated": evaluated,
        "n_skipped_no_ground_truth": skipped,
    }

    return result

In [79]:
print(evaluate_filtered_token_block)

<function evaluate_filtered_token_block at 0x000001A1FB8909A0>


In [80]:
start = perf_counter()

token_test_10k = evaluate_filtered_token_block(
    s1_df=s1_val,
    s2_index=s2_filtered_token_index,
    s3_index=s3_filtered_token_index,
    val_gt=val_gt,
    max_s1=10_000
)

print("\n10K TOKEN BLOCK TEST")
print("-" * 50)

for key, value in token_test_10k.items():
    print(f"{key}: {value}")

print(f"\nTime: {perf_counter() - start:.2f} seconds")

Processed 10,000 S1 records | candidates so far: 287,405,386

10K TOKEN BLOCK TEST
--------------------------------------------------
total_candidate_pairs: 287405386
s1_records_with_candidates: 9964
average_candidates_per_s1: 28740.5386
median_candidates_per_s1: 27352.0
p95_candidates_per_s1: 71824.49999999997
p99_candidates_per_s1: 89596.50000000001
max_candidates_per_s1: 135217
macro_candidate_recall: 0.8507466031445877
n_evaluated: 9998
n_skipped_no_ground_truth: 2

Time: 37.27 seconds


In [81]:
del s2_filtered_token_index
del s3_filtered_token_index

import gc
gc.collect()

print("25k token indexes removed from memory.")

25k token indexes removed from memory.


In [82]:
TOKEN_FREQ_THRESHOLD = 5_000

print("Testing token frequency threshold:", TOKEN_FREQ_THRESHOLD)

Testing token frequency threshold: 5000


In [83]:
start = perf_counter()

s2_filtered_token_index = build_filtered_token_index(
    S2_PATH,
    s2_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS2 5k token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Allowed tokens: 809965
Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,034,616 rows...
Finished. Index contains 809,965 tokens.

S2 5k token index built in 51.28 seconds


In [84]:
start = perf_counter()

s3_filtered_token_index = build_filtered_token_index(
    S3_PATH,
    s3_token_frequency,
    max_frequency=TOKEN_FREQ_THRESHOLD
)

print(
    f"\nS3 5k token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Allowed tokens: 862459
Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,200,000 rows...
Processed 5,285,603 rows...
Finished. Index contains 862,459 tokens.

S3 5k token index built in 50.91 seconds


In [85]:
start = perf_counter()

token_test_5k = evaluate_filtered_token_block(
    s1_df=s1_val,
    s2_index=s2_filtered_token_index,
    s3_index=s3_filtered_token_index,
    val_gt=val_gt,
    max_s1=10_000
)

print("\n10K TOKEN BLOCK TEST — 5K THRESHOLD")
print("-" * 50)

for key, value in token_test_5k.items():
    print(f"{key}: {value}")

print(f"\nTime: {perf_counter() - start:.2f} seconds")

Processed 10,000 S1 records | candidates so far: 26,629,368

10K TOKEN BLOCK TEST — 5K THRESHOLD
--------------------------------------------------
total_candidate_pairs: 26629368
s1_records_with_candidates: 8096
average_candidates_per_s1: 2662.9368
median_candidates_per_s1: 1711.0
p95_candidates_per_s1: 9285.499999999993
p99_candidates_per_s1: 14886.01
max_candidates_per_s1: 24366
macro_candidate_recall: 0.6618823223519209
n_evaluated: 9998
n_skipped_no_ground_truth: 2

Time: 3.95 seconds


In [86]:
def rarest_token_block(
    s1_df,
    s2_index,
    s3_index,
    s2_frequency,
    s3_frequency,
    val_gt,
    max_frequency=5_000,
    max_s1=None
):
    total_candidates = 0
    s1_with_candidates = 0
    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if max_s1 is not None and i >= max_s1:
            break

        s1_id = row["entity_id"]
        normalized = normalize_text(row["business_name"])

        if not normalized:
            candidate_ids = set()

        else:
            tokens = set(normalized.split())

            # Keep only tokens that exist in at least one index
            usable = []

            for token in tokens:

                freq_s2 = s2_frequency.get(token, float("inf"))
                freq_s3 = s3_frequency.get(token, float("inf"))

                freq = min(freq_s2, freq_s3)

                if freq <= max_frequency:
                    usable.append((token, freq))

            if not usable:
                candidate_ids = set()

            else:
                # Choose the rarest token
                rarest_token, _ = min(
                    usable,
                    key=lambda x: x[1]
                )

                candidate_ids = set()

                candidate_ids.update(
                    s2_index.get(rarest_token, set())
                )

                candidate_ids.update(
                    s3_index.get(rarest_token, set())
                )

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates

        if n_candidates > 0:
            s1_with_candidates += 1

        per_s1_counts.append(n_candidates)

        # Recall
        if s1_id not in val_gt:
            skipped += 1

        else:
            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:
                recall_sum += 1.0

            else:
                found = len(
                    true_matches & candidate_ids
                )

                recall_sum += (
                    found / len(true_matches)
                )

            evaluated += 1

        if (i + 1) % 10_000 == 0:
            print(
                f"Processed {i + 1:,} S1 records | "
                f"candidates so far: {total_candidates:,}"
            )

    import numpy as np

    counts = np.array(per_s1_counts)

    return {
        "total_candidate_pairs": total_candidates,
        "s1_records_with_candidates": s1_with_candidates,
        "average_candidates_per_s1": counts.mean(),
        "median_candidates_per_s1": np.median(counts),
        "p95_candidates_per_s1": np.quantile(counts, 0.95),
        "p99_candidates_per_s1": np.quantile(counts, 0.99),
        "max_candidates_per_s1": counts.max(),
        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated else 0.0
        ),
        "n_evaluated": evaluated,
        "n_skipped_no_ground_truth": skipped,
    }

In [87]:
rarest_token_test = rarest_token_block(
    s1_df=s1_val,
    s2_index=s2_filtered_token_index,
    s3_index=s3_filtered_token_index,
    s2_frequency=s2_token_frequency,
    s3_frequency=s3_token_frequency,
    val_gt=val_gt,
    max_frequency=5_000,
    max_s1=10_000
)

print("\nRAREST NAME TOKEN — 5K")
print("-" * 50)

for key, value in rarest_token_test.items():
    print(f"{key}: {value}")

Processed 10,000 S1 records | candidates so far: 12,933,582

RAREST NAME TOKEN — 5K
--------------------------------------------------
total_candidate_pairs: 12933582
s1_records_with_candidates: 8096
average_candidates_per_s1: 1293.3582
median_candidates_per_s1: 179.0
p95_candidates_per_s1: 6295.0
p99_candidates_per_s1: 8500.0
max_candidates_per_s1: 9772
macro_candidate_recall: 0.6361925812290935
n_evaluated: 9998
n_skipped_no_ground_truth: 2


In [88]:
def normalize_address(value):
    if pd.isna(value):
        return ""

    value = str(value)

    value = unicodedata.normalize("NFKC", value)
    value = value.lower()

    value = re.sub(r"[^\w\s]", " ", value)

    value = re.sub(r"\s+", " ", value).strip()

    return value

In [89]:
s1_val["address_norm"] = (
    s1_val["business_address"]
    .map(normalize_address)
)

print(
    "Validation S1 with non-empty addresses:",
    (s1_val["address_norm"] != "").sum()
)

s1_val[
    ["business_name", "business_address", "address_norm"]
].head(10)


Validation S1 with non-empty addresses: 415946


,business_name,business_address,address_norm
10,Crystal Lending PC,"11643 Prosperity Road, South Jordan, UT",11643 prosperity road south jordan ut
13,Helios,"66 Edgewood Street, Bridgeport, CT",66 edgewood street bridgeport ct
16,Smart Healthcare Private Limited,"303, 3Rd Floor Sakar 5 B/H Natraj Cinema Ashra...",303 3rd floor sakar 5 b h natraj cinema ashram...
29,Pacific Learning Laboratories LLC,"4810 Nassau Avenue, Sand Springs, OK",4810 nassau avenue sand springs ok
33,Apex Inc,"108 Richardson Street, Bethany, WV",108 richardson street bethany wv
34,Foot & Ankle Allied Center LLC,"1216 Preston Avenue, Charlottesville City, VA",1216 preston avenue charlottesville city va
37,FUE Pgim Care,"211 Dewey Road, IL, Rockton",211 dewey road il rockton
38,Primary Care Physicians Inc.,"53 Park Lane, Wellsville, NY",53 park lane wellsville ny
41,George Saul Inc,"Unit UNIT 367, 1400 Great Wolf Drive, WI, Vill...",unit unit 367 1400 great wolf drive wi village...
54,Abhushan & Brothers Co,"Hig 28 1St Floor Indra Nagar, Kanpur, Uttar Pr...",hig 28 1st floor indra nagar kanpur uttar pradesh


In [90]:
for address in s1_val["address_norm"].head(20):
    print(address)

11643 prosperity road south jordan ut
66 edgewood street bridgeport ct
303 3rd floor sakar 5 b h natraj cinema ashram road ahmedabad gujarat
4810 nassau avenue sand springs ok
108 richardson street bethany wv
1216 preston avenue charlottesville city va
211 dewey road il rockton
53 park lane wellsville ny
unit unit 367 1400 great wolf drive wi village of lake delton
hig 28 1st floor indra nagar kanpur uttar pradesh
west bengal howrah 229 kolkata netaji subhas chandra bose road vishnu enclave 3rd floor flat no 3a kolkata
a 1 malad west mumbai maharashtra 401 rahul apartment marve road
109 2 floor 16 cross road jp nagar phase 4 dollars bangalore south bangalore karnataka
1204 block c stratum venus ground nr jhansi ki rani statue nehrunagar ahmadabad city ahmedabad gujarat
28 b chand bihari nagar near khatipura bridge khatipura jaipur rajasthan
9 2113 gali no 7 kailash nagar new delhi east delhi delhi
122 dayton yellow springs road unit 4 fairborn oh
37197 sandy ridge drive north ridgevill

In [91]:
from collections import defaultdict

def build_full_address_index(path, chunk_size=200_000):

    index = defaultdict(set)
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_address"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        chunk["address_norm"] = (
            chunk["business_address"]
            .map(normalize_address)
        )

        chunk = chunk[
            chunk["address_norm"] != ""
        ]

        for address, group in chunk.groupby("address_norm"):
            index[address].update(
                group["entity_id"].tolist()
            )

        print(f"Processed {total_rows:,} rows...")

    print(
        f"Finished. Index contains "
        f"{len(index):,} normalized addresses."
    )

    return index

In [92]:
start = perf_counter()

s2_address_index = build_full_address_index(
    S2_PATH
)

print(
    f"S2 address index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,034,616 rows...
Finished. Index contains 4,286,077 normalized addresses.
S2 address index built in 336.15 seconds


In [93]:
start = perf_counter()

s3_address_index = build_full_address_index(
    S3_PATH
)

print(
    f"S3 address index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 200,000 rows...
Processed 400,000 rows...
Processed 600,000 rows...
Processed 800,000 rows...
Processed 1,000,000 rows...
Processed 1,200,000 rows...
Processed 1,400,000 rows...
Processed 1,600,000 rows...
Processed 1,800,000 rows...
Processed 2,000,000 rows...
Processed 2,200,000 rows...
Processed 2,400,000 rows...
Processed 2,600,000 rows...
Processed 2,800,000 rows...
Processed 3,000,000 rows...
Processed 3,200,000 rows...
Processed 3,400,000 rows...
Processed 3,600,000 rows...
Processed 3,800,000 rows...
Processed 4,000,000 rows...
Processed 4,200,000 rows...
Processed 4,400,000 rows...
Processed 4,600,000 rows...
Processed 4,800,000 rows...
Processed 5,000,000 rows...
Processed 5,200,000 rows...
Processed 5,285,603 rows...
Finished. Index contains 4,616,044 normalized addresses.
S3 address index built in 244.47 seconds


In [94]:
def evaluate_exact_address_block(
    s1_df,
    s2_index,
    s3_index,
    val_gt,
    max_s1=None
):

    total_candidates = 0
    s1_with_candidates = 0
    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if max_s1 is not None and i >= max_s1:
            break

        s1_id = row["entity_id"]
        address = normalize_address(
            row["business_address"]
        )

        candidate_ids = set()

        if address:
            candidate_ids.update(
                s2_index.get(address, set())
            )

            candidate_ids.update(
                s3_index.get(address, set())
            )

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates

        if n_candidates > 0:
            s1_with_candidates += 1

        per_s1_counts.append(n_candidates)

        # Candidate recall
        if s1_id not in val_gt:
            skipped += 1

        else:
            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:
                recall_sum += 1.0

            else:
                found = len(
                    true_matches & candidate_ids
                )

                recall_sum += (
                    found / len(true_matches)
                )

            evaluated += 1

        if (i + 1) % 10_000 == 0:
            print(
                f"Processed {i + 1:,} S1 records | "
                f"candidates so far: "
                f"{total_candidates:,}"
            )

    import numpy as np

    counts = np.array(per_s1_counts)

    return {
        "total_candidate_pairs": total_candidates,
        "s1_records_with_candidates": s1_with_candidates,
        "average_candidates_per_s1": counts.mean(),
        "median_candidates_per_s1": np.median(counts),
        "p95_candidates_per_s1": np.quantile(counts, 0.95),
        "p99_candidates_per_s1": np.quantile(counts, 0.99),
        "max_candidates_per_s1": counts.max(),
        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated else 0.0
        ),
        "n_evaluated": evaluated,
        "n_skipped_no_ground_truth": skipped
    }

In [95]:
address_test_10k = evaluate_exact_address_block(
    s1_df=s1_val,
    s2_index=s2_address_index,
    s3_index=s3_address_index,
    val_gt=val_gt,
    max_s1=10_000
)

print("\nEXACT NORMALIZED ADDRESS - 10K TEST")
print("-" * 55)

for key, value in address_test_10k.items():
    print(f"{key}: {value}")

Processed 10,000 S1 records | candidates so far: 3,367

EXACT NORMALIZED ADDRESS - 10K TEST
-------------------------------------------------------
total_candidate_pairs: 3367
s1_records_with_candidates: 2373
average_candidates_per_s1: 0.3367
median_candidates_per_s1: 0.0
p95_candidates_per_s1: 2.0
p99_candidates_per_s1: 3.0
max_candidates_per_s1: 13
macro_candidate_recall: 0.13180100666598035
n_evaluated: 9998
n_skipped_no_ground_truth: 2


In [96]:
def extract_address_numbers(value):
    if pd.isna(value):
        return set()

    text = str(value).lower()

    return set(
        re.findall(r"\b\d+[a-z]?\b", text)
    )

In [97]:
for address in s1_val["business_address"].head(20):
    print(
        address,
        "→",
        extract_address_numbers(address)
    )

11643 Prosperity Road, South Jordan, UT → {'11643'}
66 Edgewood Street, Bridgeport, CT → {'66'}
303, 3Rd Floor Sakar 5 B/H Natraj Cinema Ashram Road, Ahmedabad, Gujarat → {'303', '5'}
4810 Nassau Avenue, Sand Springs, OK → {'4810'}
108 Richardson Street, Bethany, WV → {'108'}
1216 Preston Avenue, Charlottesville City, VA → {'1216'}
211 Dewey Road, IL, Rockton → {'211'}
53 Park Lane, Wellsville, NY → {'53'}
Unit UNIT 367, 1400 Great Wolf Drive, WI, Village Of Lake Delton → {'367', '1400'}
Hig 28 1St Floor Indra Nagar, Kanpur, Uttar Pradesh → {'28'}
West Bengal, Howrah, 229, Kolkata, Netaji Subhas Chandra Bose Road Vishnu Enclave, 3Rd Floor, Flat No.- 3A, Kolkata → {'3a', '229'}
A-1, Malad (West), Mumbai, Maharashtra, 401, Rahul Apartment, Marve Road → {'1', '401'}
109 2 Floor 16 Cross Road, Jp Nagar Phase 4, Dollars, Bangalore South, Bangalore, Karnataka → {'109', '16', '4', '2'}
1204, Block-C, Stratum @ Venus Ground, Nr. Jhansi Ki Rani Statue, Nehrunagar, Ahmadabad City, Ahmedabad, Guj

In [98]:
s1_val["address_numbers"] = (
    s1_val["business_address"]
    .map(extract_address_numbers)
)

print(
    "S1 records with at least one address number:",
    (s1_val["address_numbers"].map(len) > 0).sum()
)

s1_val[
    ["business_address", "address_numbers"]
].head(20)

S1 records with at least one address number: 395773


,business_address,address_numbers
10,"11643 Prosperity Road, South Jordan, UT",{11643}
13,"66 Edgewood Street, Bridgeport, CT",{66}
16,"303, 3Rd Floor Sakar 5 B/H Natraj Cinema Ashra...","{303, 5}"
29,"4810 Nassau Avenue, Sand Springs, OK",{4810}
33,"108 Richardson Street, Bethany, WV",{108}
34,"1216 Preston Avenue, Charlottesville City, VA",{1216}
37,"211 Dewey Road, IL, Rockton",{211}
38,"53 Park Lane, Wellsville, NY",{53}
41,"Unit UNIT 367, 1400 Great Wolf Drive, WI, Vill...","{367, 1400}"
54,"Hig 28 1St Floor Indra Nagar, Kanpur, Uttar Pr...",{28}


In [99]:
from collections import Counter

def build_address_number_frequency(path, chunk_size=200_000):

    frequency = Counter()
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["business_address"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        for address in chunk["business_address"]:
            numbers = extract_address_numbers(address)

            for number in numbers:
                frequency[number] += 1

        if total_rows % 1_000_000 == 0:
            print(f"Processed {total_rows:,} rows...")

    print(
        f"Finished. Unique address numbers: "
        f"{len(frequency):,}"
    )

    return frequency

In [100]:
start = perf_counter()

s2_address_number_frequency = build_address_number_frequency(
    S2_PATH
)

print(
    f"S2 number frequency built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Finished. Unique address numbers: 96,977
S2 number frequency built in 32.16 seconds


In [101]:
start = perf_counter()

s3_address_number_frequency = build_address_number_frequency(
    S3_PATH
)

print(
    f"S3 number frequency built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Finished. Unique address numbers: 97,660
S3 number frequency built in 35.04 seconds


In [102]:
print("TOP S2 ADDRESS NUMBERS")

for number, count in s2_address_number_frequency.most_common(30):
    print(repr(number), "→", count)


print("\nTOP S3 ADDRESS NUMBERS")

for number, count in s3_address_number_frequency.most_common(30):
    print(repr(number), "→", count)

TOP S2 ADDRESS NUMBERS
'1' → 281590
'2' → 228480
'3' → 121045
'4' → 98543
'5' → 87501
'6' → 79997
'8' → 73792
'7' → 70463
'9' → 61552
'10' → 56686
'11' → 52792
'12' → 49938
'14' → 42531
'15' → 40092
'13' → 39186
'16' → 37420
'17' → 33730
'24' → 32883
'18' → 32401
'19' → 31061
'20' → 30705
'22' → 29393
'21' → 28651
'23' → 27536
'25' → 25124
'30' → 21987
'27' → 21747
'26' → 21705
'28' → 21388
'31' → 20254

TOP S3 ADDRESS NUMBERS
'1' → 309505
'2' → 238180
'3' → 126735
'4' → 100879
'5' → 89308
'6' → 81292
'8' → 75037
'7' → 71440
'9' → 62370
'10' → 57521
'11' → 52646
'12' → 50569
'14' → 43001
'15' → 40558
'13' → 39900
'24' → 38838
'16' → 37831
'17' → 34137
'18' → 32988
'20' → 31239
'19' → 31160
'22' → 29757
'21' → 29109
'23' → 27543
'0' → 27451
'25' → 25897
'26' → 22339
'30' → 22079
'27' → 22041
'28' → 21786


In [103]:
thresholds = [100, 500, 1000, 5000, 10000, 25000]

print("S2 address numbers surviving each threshold")
print("-" * 50)

for threshold in thresholds:
    count = sum(
        freq <= threshold
        for freq in s2_address_number_frequency.values()
    )
    print(f"<= {threshold:>5}: {count:,} numbers")


print("\nS3 address numbers surviving each threshold")
print("-" * 50)

for threshold in thresholds:
    count = sum(
        freq <= threshold
        for freq in s3_address_number_frequency.values()
    )
    print(f"<= {threshold:>5}: {count:,} numbers")

S2 address numbers surviving each threshold
--------------------------------------------------
<=   100: 91,046 numbers
<=   500: 95,535 numbers
<=  1000: 96,158 numbers
<=  5000: 96,796 numbers
<= 10000: 96,898 numbers
<= 25000: 96,952 numbers

S3 address numbers surviving each threshold
--------------------------------------------------
<=   100: 91,504 numbers
<=   500: 96,144 numbers
<=  1000: 96,806 numbers
<=  5000: 97,470 numbers
<= 10000: 97,568 numbers
<= 25000: 97,634 numbers


In [104]:
for threshold in thresholds:

    usable = 0

    for numbers in s1_val["address_numbers"]:

        if any(
            min(
                s2_address_number_frequency.get(
                    number,
                    float("inf")
                ),
                s3_address_number_frequency.get(
                    number,
                    float("inf")
                )
            ) <= threshold
            for number in numbers
        ):
            usable += 1

    print(
        f"Threshold {threshold:>5}: "
        f"{usable:,} / {len(s1_val):,} S1 records"
    )

Threshold   100: 55,857 / 415,946 S1 records
Threshold   500: 133,505 / 415,946 S1 records
Threshold  1000: 166,710 / 415,946 S1 records
Threshold  5000: 262,999 / 415,946 S1 records
Threshold 10000: 312,454 / 415,946 S1 records
Threshold 25000: 353,480 / 415,946 S1 records


In [105]:
from collections import defaultdict

def build_name_number_index(
    path,
    name_frequency,
    number_frequency,
    max_name_frequency=5000,
    max_number_frequency=5000,
    chunk_size=200_000
):
    index = defaultdict(set)
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name", "business_address"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        for _, row in chunk.iterrows():

            entity_id = row["entity_id"]

            name = normalize_text(row["business_name"])
            address = row["business_address"]

            if not name:
                continue

            tokens = set(name.split())
            numbers = extract_address_numbers(address)

            usable_tokens = [
                token
                for token in tokens
                if name_frequency.get(
                    token,
                    float("inf")
                ) <= max_name_frequency
            ]

            usable_numbers = [
                number
                for number in numbers
                if number_frequency.get(
                    number,
                    float("inf")
                ) <= max_number_frequency
            ]

            for token in usable_tokens:
                for number in usable_numbers:

                    key = (token, number)

                    index[key].add(entity_id)

        if total_rows % 1_000_000 == 0:
            print(
                f"Processed {total_rows:,} rows..."
            )

    print(
        f"Finished. Index contains "
        f"{len(index):,} (token, number) keys."
    )

    return index

In [106]:
start = perf_counter()

s2_name_number_index = build_name_number_index(
    S2_PATH,
    s2_token_frequency,
    s2_address_number_frequency,
    max_name_frequency=5000,
    max_number_frequency=5000
)

print(
    f"S2 name+number index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Finished. Index contains 3,546,658 (token, number) keys.
S2 name+number index built in 441.73 seconds


In [107]:
start = perf_counter()

s3_name_number_index = build_name_number_index(
    S3_PATH,
    s3_token_frequency,
    s3_address_number_frequency,
    max_name_frequency=5000,
    max_number_frequency=5000
)

print(
    f"S3 name+number index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Finished. Index contains 3,678,398 (token, number) keys.
S3 name+number index built in 321.79 seconds


In [111]:
def evaluate_name_number_block(
    s1_df,
    s2_index,
    s3_index,
    val_gt,
    name_frequency,
    number_frequency,
    max_name_frequency=5000,
    max_number_frequency=5000,
    max_s1=None
):
    import numpy as np

    total_candidates = 0
    s1_with_candidates = 0
    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if max_s1 is not None and i >= max_s1:
            break

        s1_id = row["entity_id"]

        name = normalize_text(row["business_name"])

        numbers = extract_address_numbers(
            row["business_address"]
        )

        candidate_ids = set()

        # -----------------------------------------
        # NAME + ADDRESS NUMBER BLOCK
        # -----------------------------------------

        if name and numbers:

            tokens = set(name.split())

            usable_tokens = [
                token
                for token in tokens
                if name_frequency.get(
                    token,
                    float("inf")
                ) <= max_name_frequency
            ]

            usable_numbers = [
                number
                for number in numbers
                if number_frequency.get(
                    number,
                    float("inf")
                ) <= max_number_frequency
            ]

            for token in usable_tokens:

                for number in usable_numbers:

                    key = (token, number)

                    candidate_ids.update(
                        s2_index.get(key, set())
                    )

                    candidate_ids.update(
                        s3_index.get(key, set())
                    )

        # -----------------------------------------
        # CANDIDATE COUNT
        # -----------------------------------------

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates

        if n_candidates > 0:
            s1_with_candidates += 1

        per_s1_counts.append(n_candidates)

        # -----------------------------------------
        # CANDIDATE RECALL
        # -----------------------------------------

        if s1_id not in val_gt:

            skipped += 1

        else:

            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:

                recall_sum += 1.0

            else:

                found = len(
                    true_matches & candidate_ids
                )

                recall_sum += (
                    found / len(true_matches)
                )

            evaluated += 1

        # -----------------------------------------
        # PROGRESS
        # -----------------------------------------

        if (i + 1) % 10_000 == 0:

            print(
                f"Processed {i + 1:,} S1 records | "
                f"candidates so far: "
                f"{total_candidates:,}"
            )

    # -----------------------------------------
    # SUMMARY STATISTICS
    # -----------------------------------------

    counts = np.array(per_s1_counts)

    return {
        "total_candidate_pairs": total_candidates,

        "s1_records_with_candidates":
            s1_with_candidates,

        "average_candidates_per_s1":
            counts.mean(),

        "median_candidates_per_s1":
            np.median(counts),

        "p95_candidates_per_s1":
            np.quantile(counts, 0.95),

        "p99_candidates_per_s1":
            np.quantile(counts, 0.99),

        "max_candidates_per_s1":
            counts.max(),

        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated
            else 0.0
        ),

        "n_evaluated":
            evaluated,

        "n_skipped_no_ground_truth":
            skipped
    }

In [112]:
name_number_test = evaluate_name_number_block(
    s1_df=s1_val,
    s2_index=s2_name_number_index,
    s3_index=s3_name_number_index,
    val_gt=val_gt,
    name_frequency=s2_token_frequency,
    number_frequency=s2_address_number_frequency,
    max_name_frequency=5000,
    max_number_frequency=5000,
    max_s1=10_000
)

print("\nNAME TOKEN + ADDRESS NUMBER — 10K TEST")
print("-" * 55)

for key, value in name_number_test.items():
    print(f"{key}: {value}")

Processed 10,000 S1 records | candidates so far: 15,734

NAME TOKEN + ADDRESS NUMBER — 10K TEST
-------------------------------------------------------
total_candidate_pairs: 15734
s1_records_with_candidates: 4508
average_candidates_per_s1: 1.5734
median_candidates_per_s1: 0.0
p95_candidates_per_s1: 6.0
p99_candidates_per_s1: 11.0
max_candidates_per_s1: 26
macro_candidate_recall: 0.3489065489865643
n_evaluated: 9998
n_skipped_no_ground_truth: 2


COMPARISON TABLE

In [115]:
blocking_results = []

blocking_results.append({
    "block": "Exact normalized name",
    "recall": exact_name_recall["macro_candidate_recall"],
    "avg_candidates": 13.968,
    "median_candidates": 2.0,
    "p95_candidates": 78.0,
    "p99_candidates": 200.0,
    "max_candidates": 459
})

blocking_results.append({
    "block": "Name token <= 5k",
    "recall": 0.6618823223519209,
    "avg_candidates": 2662.9368,
    "median_candidates": 1711.0,
    "p95_candidates": 9285.5,
    "p99_candidates": 14886.01,
    "max_candidates": 24366
})

blocking_results.append({
    "block": "Name token <= 25k",
    "recall": 0.8507466031445877,
    "avg_candidates": 28740.5386,
    "median_candidates": 27352.0,
    "p95_candidates": 71824.5,
    "p99_candidates": 89596.5,
    "max_candidates": 135217
})

blocking_results.append({
    "block": "Rarest name token <= 5k",
    "recall": 0.6361525732274932,
    "avg_candidates": 1293.3583,
    "median_candidates": 179.0,
    "p95_candidates": 6295.0,
    "p99_candidates": 8500.0,
    "max_candidates": 9772
})

blocking_results.append({
    "block": "Exact normalized address",
    "recall": 0.13180100666598035,
    "avg_candidates": 0.3367,
    "median_candidates": 0.0,
    "p95_candidates": 2.0,
    "p99_candidates": 3.0,
    "max_candidates": 13
})

blocking_results.append({
    "block": "Name token + address number",
    "recall": 0.3489065489865643,
    "avg_candidates": 1.5734,
    "median_candidates": 0.0,
    "p95_candidates": 6.0,
    "p99_candidates": 11.0,
    "max_candidates": 26
})

blocking_results_df = pd.DataFrame(blocking_results)

blocking_results_df

,block,recall,avg_candidates,median_candidates,p95_candidates,p99_candidates,max_candidates
0,Exact normalized name,0.262762,13.9680,2.0,78.0,200.00,459
1,Name token <= 5k,0.661882,2662.9368,1711.0,9285.5,14886.01,24366
2,Name token <= 25k,0.850747,28740.5386,27352.0,71824.5,89596.50,135217
3,Rarest name token <= 5k,0.636153,1293.3583,179.0,6295.0,8500.00,9772
4,Exact normalized address,0.131801,0.3367,0.0,2.0,3.00,13
5,Name token + address number,0.348907,1.5734,0.0,6.0,11.00,26


In [116]:
from collections import Counter

def build_address_token_frequency(s1_df):
    counter = Counter()

    for address in s1_df["address_norm"]:
        if not address:
            continue

        tokens = set(address.split())
        counter.update(tokens)

    return counter

In [117]:
s1_address_frequency = build_address_token_frequency(s1_val)

print("Unique S1 address tokens:", len(s1_address_frequency))

print("\nTOP S1 ADDRESS TOKENS")
for token, count in s1_address_frequency.most_common(30):
    print(repr(token), "→", count)

Unique S1 address tokens: 171291

TOP S1 ADDRESS TOKENS
'road' → 86673
'no' → 68298
'street' → 58589
'drive' → 41659
'maharashtra' → 36134
'unit' → 36128
'avenue' → 35684
'floor' → 30725
'nagar' → 29572
'city' → 27757
'west' → 25064
'tx' → 24798
'1' → 24630
'delhi' → 23372
'pradesh' → 21308
'new' → 21076
'lane' → 20380
'c' → 19611
'ny' → 19185
'nc' → 18048
'2' → 17552
'a' → 17452
'plot' → 16461
'mumbai' → 16427
'b' → 16052
'oh' → 15945
'il' → 14250
'uttar' → 13928
'o' → 13476
'karnataka' → 13186


In [118]:
ADDRESS_TOKEN_FREQ_THRESHOLD = 5_000

useful_address_tokens = {
    token
    for token, count in s1_address_frequency.items()
    if count <= ADDRESS_TOKEN_FREQ_THRESHOLD
}

print("Total unique address tokens:", len(s1_address_frequency))
print(
    f"Useful address tokens (frequency <= {ADDRESS_TOKEN_FREQ_THRESHOLD}):",
    len(useful_address_tokens)
)

print("\nSample useful address tokens:")
print(list(useful_address_tokens)[:50])

Total unique address tokens: 171291
Useful address tokens (frequency <= 5000): 171194

Sample useful address tokens:
['kayampur', 'mahendergarh', 'local', 'sankhavaram', 'kodamullil', 'trikampura', '18287', 'hardaway', 'niranjan', '56rajalakshmi', '161st', 'barkeater', 'bonacker', 'birsi', 'decaro', 'baer', 'quarts', 'bridlepath', 'bellville', 'hinde', 'mast', 'taharabad', '9268', 'margutta', 'damaji', 'plotno766kamalkunjnrwatertanknewnandanvan', 'vrindavanvihar', '3069', 'sorale', '5296', 'bradshire', '26310', 'viswapuram', 'sidhhpur', 'ramsundar', 'doaba', 'tawde', '3723', 'alkapur', 'jagadeshnagar', 'mahat', 'mcorp', 'patliputra', '1795', 'idgha', 'yojan', 'festive', 'vilathikulam', 'bayside', 'dattatray']


In [119]:
usable_s1 = 0
total_usable_tokens = 0

for address in s1_val["address_norm"]:
    if not address:
        continue

    tokens = set(address.split())
    useful = tokens & useful_address_tokens

    if useful:
        usable_s1 += 1
        total_usable_tokens += len(useful)

print(
    f"S1 records with at least one useful address token: "
    f"{usable_s1:,} / {len(s1_val):,}"
)

print(
    "Average useful address tokens per S1:",
    total_usable_tokens / len(s1_val)
)

S1 records with at least one useful address token: 415,941 / 415,946
Average useful address tokens per S1: 4.9068677184057545


In [120]:
from collections import defaultdict
from time import perf_counter


def build_filtered_address_token_index(
    path,
    useful_tokens,
    chunk_size=200_000
):
    index = defaultdict(set)
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_address"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        chunk["address_norm"] = (
            chunk["business_address"]
            .fillna("")
            .map(normalize_text)
        )

        for _, row in chunk.iterrows():

            address = row["address_norm"]

            if not address:
                continue

            tokens = set(address.split()) & useful_tokens

            for token in tokens:
                index[token].add(row["entity_id"])

        if total_rows % 1_000_000 < chunk_size:
            print(f"Processed {total_rows:,} rows...")

    print(f"Finished. Index contains {len(index):,} tokens.")

    return index

In [121]:
start = perf_counter()

s2_address_token_index = build_filtered_address_token_index(
    S2_PATH,
    useful_address_tokens
)

print(
    f"S2 address-token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Processed 5,034,616 rows...
Finished. Index contains 165,364 tokens.
S2 address-token index built in 338.07 seconds


In [122]:
start = perf_counter()

s3_address_token_index = build_filtered_address_token_index(
    S3_PATH,
    useful_address_tokens
)

print(
    f"S3 address-token index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Finished. Index contains 163,073 tokens.
S3 address-token index built in 313.60 seconds


In [123]:
def evaluate_address_token_block(
    s1_df,
    s2_index,
    s3_index,
    val_gt,
    max_s1=10_000
):
    total_candidates = 0
    s1_with_candidates = 0
    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if i >= max_s1:
            break

        s1_id = row["entity_id"]
        address = row["address_norm"]

        candidate_ids = set()

        if address:
            tokens = set(address.split())

            for token in tokens:
                candidate_ids.update(
                    s2_index.get(token, set())
                )
                candidate_ids.update(
                    s3_index.get(token, set())
                )

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates
        per_s1_counts.append(n_candidates)

        if n_candidates > 0:
            s1_with_candidates += 1

        # Candidate recall
        if s1_id not in val_gt:
            skipped += 1
        else:
            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:
                recall_sum += 1.0
            else:
                found = len(true_matches & candidate_ids)
                recall_sum += found / len(true_matches)

            evaluated += 1

        if (i + 1) % 2_000 == 0:
            print(
                f"Processed {i+1:,} S1 records | "
                f"candidates so far: {total_candidates:,}"
            )

    counts = np.array(per_s1_counts)

    return {
        "total_candidate_pairs": total_candidates,
        "s1_records_with_candidates": s1_with_candidates,
        "average_candidates_per_s1": counts.mean(),
        "median_candidates_per_s1": np.median(counts),
        "p95_candidates_per_s1": np.quantile(counts, 0.95),
        "p99_candidates_per_s1": np.quantile(counts, 0.99),
        "max_candidates_per_s1": counts.max(),
        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated > 0 else 0.0
        ),
        "n_evaluated": evaluated,
        "n_skipped_no_ground_truth": skipped
    }

In [124]:
start = perf_counter()

address_token_test = evaluate_address_token_block(
    s1_val,
    s2_address_token_index,
    s3_address_token_index,
    val_gt,
    max_s1=10_000
)

print("\nADDRESS-TOKEN — 10K TEST")
print("-" * 55)

for key, value in address_token_test.items():
    print(f"{key}: {value}")

print(f"\nTime: {perf_counter() - start:.2f} seconds")

Processed 2,000 S1 records | candidates so far: 195,663,211
Processed 4,000 S1 records | candidates so far: 400,984,837
Processed 6,000 S1 records | candidates so far: 593,740,107
Processed 8,000 S1 records | candidates so far: 792,619,801
Processed 10,000 S1 records | candidates so far: 991,312,196

ADDRESS-TOKEN — 10K TEST
-------------------------------------------------------
total_candidate_pairs: 991312196
s1_records_with_candidates: 10000
average_candidates_per_s1: 99131.2196
median_candidates_per_s1: 62162.0
p95_candidates_per_s1: 284959.0999999995
p99_candidates_per_s1: 741608.5900000002
max_candidates_per_s1: 1958800
macro_candidate_recall: 0.9553388816291395
n_evaluated: 9998
n_skipped_no_ground_truth: 2

Time: 133.87 seconds


In [125]:
def evaluate_address_token_number_block(
    s1_df,
    s2_name_number_index,
    s3_name_number_index,
    s2_address_token_index,
    s3_address_token_index,
    val_gt,
    max_s1=10_000
):
    total_candidates = 0
    s1_with_candidates = 0
    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if i >= max_s1:
            break

        s1_id = row["entity_id"]

        # -------------------------
        # Address tokens
        # -------------------------
        address = row["address_norm"]

        token_candidates = set()

        if address:
            tokens = set(address.split())

            for token in tokens:
                token_candidates.update(
                    s2_address_token_index.get(token, set())
                )
                token_candidates.update(
                    s3_address_token_index.get(token, set())
                )

        # -------------------------
        # Address numbers
        # -------------------------
        numbers = row["address_numbers"]

        number_candidates = set()

        if numbers:
            for number in numbers:
                number_candidates.update(
                    s2_name_number_index.get(number, set())
                )
                number_candidates.update(
                    s3_name_number_index.get(number, set())
                )

        # -------------------------
        # INTERSECTION
        # -------------------------
        candidate_ids = (
            token_candidates
            & number_candidates
        )

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates
        per_s1_counts.append(n_candidates)

        if n_candidates > 0:
            s1_with_candidates += 1

        # -------------------------
        # Recall
        # -------------------------
        if s1_id not in val_gt:
            skipped += 1

        else:
            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:
                recall_sum += 1.0

            else:
                found = len(
                    true_matches & candidate_ids
                )

                recall_sum += (
                    found / len(true_matches)
                )

            evaluated += 1

        if (i + 1) % 2_000 == 0:
            print(
                f"Processed {i+1:,} S1 records | "
                f"candidates so far: {total_candidates:,}"
            )

    counts = np.array(per_s1_counts)

    return {
        "total_candidate_pairs": total_candidates,
        "s1_records_with_candidates": s1_with_candidates,
        "average_candidates_per_s1": counts.mean(),
        "median_candidates_per_s1": np.median(counts),
        "p95_candidates_per_s1": np.quantile(counts, 0.95),
        "p99_candidates_per_s1": np.quantile(counts, 0.99),
        "max_candidates_per_s1": counts.max(),
        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated > 0 else 0.0
        ),
        "n_evaluated": evaluated,
        "n_skipped_no_ground_truth": skipped
    }

In [126]:
start = perf_counter()

address_token_number_test = evaluate_address_token_number_block(
    s1_val,
    s2_name_number_index,
    s3_name_number_index,
    s2_address_token_index,
    s3_address_token_index,
    val_gt,
    max_s1=10_000
)

print("\nADDRESS TOKEN + ADDRESS NUMBER — 10K TEST")
print("-" * 60)

for key, value in address_token_number_test.items():
    print(f"{key}: {value}")

print(f"\nTime: {perf_counter() - start:.2f} seconds")

Processed 2,000 S1 records | candidates so far: 0
Processed 4,000 S1 records | candidates so far: 0
Processed 6,000 S1 records | candidates so far: 0
Processed 8,000 S1 records | candidates so far: 0
Processed 10,000 S1 records | candidates so far: 0

ADDRESS TOKEN + ADDRESS NUMBER — 10K TEST
------------------------------------------------------------
total_candidate_pairs: 0
s1_records_with_candidates: 0
average_candidates_per_s1: 0.0
median_candidates_per_s1: 0.0
p95_candidates_per_s1: 0.0
p99_candidates_per_s1: 0.0
max_candidates_per_s1: 0
macro_candidate_recall: 0.05731146229245849
n_evaluated: 9998
n_skipped_no_ground_truth: 2

Time: 128.73 seconds


In [127]:
from collections import defaultdict
from time import perf_counter

def build_address_number_index(
    path,
    number_frequency,
    max_number_frequency=5000,
    chunk_size=200_000
):
    index = defaultdict(set)
    total_rows = 0

    allowed_numbers = {
        number
        for number, count in number_frequency.items()
        if count <= max_number_frequency
    }

    print("Allowed address numbers:", len(allowed_numbers))

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_address"],
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        chunk["address_norm"] = (
            chunk["business_address"]
            .fillna("")
            .map(normalize_text)
        )

        for _, row in chunk.iterrows():

            address = row["address_norm"]

            if not address:
                continue

            numbers = set(
                re.findall(r"\d+", address)
            )

            numbers &= allowed_numbers

            for number in numbers:
                index[number].add(row["entity_id"])

        if total_rows % 1_000_000 < chunk_size:
            print(f"Processed {total_rows:,} rows...")

    print(
        f"Finished. Index contains "
        f"{len(index):,} address numbers."
    )

    return index

In [128]:
start = perf_counter()

s2_address_number_index = build_address_number_index(
    S2_PATH,
    s2_address_number_frequency,
    max_number_frequency=5000
)

print(
    f"S2 address-number index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Allowed address numbers: 96796
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Processed 5,034,616 rows...
Finished. Index contains 71,945 address numbers.
S2 address-number index built in 226.32 seconds


In [129]:
start = perf_counter()

s3_address_number_index = build_address_number_index(
    S3_PATH,
    s3_address_number_frequency,
    max_number_frequency=5000
)

print(
    f"S3 address-number index built in "
    f"{perf_counter() - start:.2f} seconds"
)

Allowed address numbers: 97470
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Finished. Index contains 72,281 address numbers.
S3 address-number index built in 229.80 seconds


In [130]:
def evaluate_address_number_block(
    s1_df,
    s2_index,
    s3_index,
    val_gt,
    max_s1=None
):
    total_candidates = 0
    s1_records_with_candidates = 0
    per_s1_counts = []

    recall_sum = 0.0
    evaluated = 0
    skipped = 0

    for i, (_, row) in enumerate(s1_df.iterrows()):

        if max_s1 is not None and i >= max_s1:
            break

        s1_id = row["entity_id"]
        numbers = row["address_numbers"]

        candidate_ids = set()

        for number in numbers:
            candidate_ids.update(s2_index.get(number, set()))
            candidate_ids.update(s3_index.get(number, set()))

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates

        if n_candidates > 0:
            s1_records_with_candidates += 1

        per_s1_counts.append(n_candidates)

        if s1_id not in val_gt:
            skipped += 1
        else:
            true_matches = val_gt[s1_id]

            if len(true_matches) == 0:
                recall_sum += 1.0
            else:
                found = len(true_matches & candidate_ids)
                recall_sum += found / len(true_matches)

            evaluated += 1

        if (i + 1) % 2000 == 0:
            print(
                f"Processed {i + 1:,} S1 records | "
                f"candidates so far: {total_candidates:,}"
            )

    import numpy as np

    counts = np.array(per_s1_counts)

    return {
        "total_candidate_pairs": total_candidates,
        "s1_records_with_candidates": s1_records_with_candidates,
        "average_candidates_per_s1": counts.mean(),
        "median_candidates_per_s1": np.median(counts),
        "p95_candidates_per_s1": np.quantile(counts, 0.95),
        "p99_candidates_per_s1": np.quantile(counts, 0.99),
        "max_candidates_per_s1": counts.max(),
        "macro_candidate_recall": (
            recall_sum / evaluated
            if evaluated > 0 else 0.0
        ),
        "n_evaluated": evaluated,
        "n_skipped_no_ground_truth": skipped
    }

In [131]:
start = perf_counter()

address_number_test = evaluate_address_number_block(
    s1_val,
    s2_address_number_index,
    s3_address_number_index,
    val_gt,
    max_s1=10_000
)

print("\nADDRESS NUMBER — 10K TEST")
print("-" * 60)

for key, value in address_number_test.items():
    print(f"{key}: {value}")

print(f"\nTime: {perf_counter() - start:.2f} seconds")

Processed 2,000 S1 records | candidates so far: 3,530,203
Processed 4,000 S1 records | candidates so far: 6,799,078
Processed 6,000 S1 records | candidates so far: 9,955,869
Processed 8,000 S1 records | candidates so far: 13,113,241
Processed 10,000 S1 records | candidates so far: 16,668,036

ADDRESS NUMBER — 10K TEST
------------------------------------------------------------
total_candidate_pairs: 16668036
s1_records_with_candidates: 6069
average_candidates_per_s1: 1666.8036
median_candidates_per_s1: 201.5
p95_candidates_per_s1: 8259.449999999993
p99_candidates_per_s1: 10392.02
max_candidates_per_s1: 26427
macro_candidate_recall: 0.49492818837938063
n_evaluated: 9998
n_skipped_no_ground_truth: 2

Time: 4.33 seconds


In [147]:
def generate_combined_candidates(
    row,
    s2_name_index,
    s3_name_index,
    s2_name_token_index,
    s3_name_token_index,
    s2_address_number_index,
    s3_address_number_index,
    s2_name_number_index,
    s3_name_number_index,
    s2_token_frequency,
    s3_token_frequency,
    s2_address_number_frequency,
    s3_address_number_frequency,
    max_name_frequency=5000,
    max_number_frequency=5000
):

    candidate_ids = set()

    name = normalize_text(row["business_name"])

    numbers = extract_address_numbers(
        row["business_address"]
    )

    # =========================================
    # 1. EXACT NORMALIZED NAME
    # =========================================

    if name:

        candidate_ids.update(
            s2_name_index.get(name, set())
        )

        candidate_ids.update(
            s3_name_index.get(name, set())
        )
    if name:

        candidate_ids.update(
            s2_full_name_index.get(name, set())
        )

        candidate_ids.update(
            s3_full_name_index.get(name, set())
        )

    # =========================================
    # 2. RARE NAME TOKEN
    # =========================================

    tokens = set(name.split()) if name else set()

    usable_tokens = [
        token
        for token in tokens
        if (
            s2_token_frequency.get(
                token,
                float("inf")
            ) <= max_name_frequency
            or
            s3_token_frequency.get(
                token,
                float("inf")
            ) <= max_name_frequency
        )
    ]

    for token in usable_tokens:

        candidate_ids.update(
            s2_name_token_index.get(token, set())
        )

        candidate_ids.update(
            s3_name_token_index.get(token, set())
        )

    # =========================================
    # 3. ADDRESS NUMBER
    # =========================================

    usable_numbers = [
        number
        for number in numbers
        if (
            s2_address_number_frequency.get(
                number,
                float("inf")
            ) <= max_number_frequency
            or
            s3_address_number_frequency.get(
                number,
                float("inf")
            ) <= max_number_frequency
        )
    ]

    for number in usable_numbers:

        candidate_ids.update(
            s2_address_number_index.get(
                number,
                set()
            )
        )

        candidate_ids.update(
            s3_address_number_index.get(
                number,
                set()
            )
        )

    # =========================================
    # 4. NAME TOKEN + ADDRESS NUMBER
    # =========================================

    for token in usable_tokens:

        for number in usable_numbers:

            key = (token, number)

            candidate_ids.update(
                s2_name_number_index.get(
                    key,
                    set()
                )
            )

            candidate_ids.update(
                s3_name_number_index.get(
                    key,
                    set()
                )
            )

    return candidate_ids

In [148]:
[name for name in globals() if "frequency" in name.lower()]

['token_frequency_s2',
 'token_frequency_s3',
 'token_frequency_full',
 's2_token_frequency',
 's3_token_frequency',
 'build_address_number_frequency',
 's2_address_number_frequency',
 's3_address_number_frequency',
 'build_address_token_frequency',
 's1_address_frequency']

In [149]:
from time import perf_counter
import numpy as np

start = perf_counter()

combined_counts = []
combined_recall_sum = 0.0
combined_evaluated = 0
combined_skipped = 0

for i, (_, row) in enumerate(s1_val.iterrows()):

    if i >= 10_000:
        break

    s1_id = row["entity_id"]

    # Generate UNION of all blocking methods
    candidate_ids = generate_combined_candidates(
        row,

        s2_name_index,
        s3_name_index,

        s2_name_token_index,
        s3_name_token_index,

        s2_address_number_index,
        s3_address_number_index,

        s2_name_number_index,
        s3_name_number_index,

        s2_token_frequency,
        s3_token_frequency,
        s2_address_number_frequency,
        s3_address_number_frequency,

        max_name_frequency=5000,
        max_number_frequency=5000
    )

    # -----------------------------------------
    # Candidate count
    # -----------------------------------------

    n_candidates = len(candidate_ids)

    combined_counts.append(n_candidates)

    # -----------------------------------------
    # Candidate recall
    # -----------------------------------------

    if s1_id not in val_gt:

        combined_skipped += 1

    else:

        true_matches = val_gt[s1_id]

        if len(true_matches) == 0:

            combined_recall_sum += 1.0

        else:

            found = len(
                true_matches & candidate_ids
            )

            combined_recall_sum += (
                found / len(true_matches)
            )

        combined_evaluated += 1

    # -----------------------------------------
    # Progress
    # -----------------------------------------

    if (i + 1) % 2_000 == 0:

        print(
            f"Processed {i + 1:,} S1 records | "
            f"candidates so far: "
            f"{sum(combined_counts):,}"
        )


# =========================================
# RESULTS
# =========================================

counts = np.array(combined_counts)

print("\nCOMBINED BLOCKER — 10K TEST")
print("-" * 60)

print(
    "total_candidate_pairs:",
    counts.sum()
)

print(
    "s1_records_with_candidates:",
    np.sum(counts > 0)
)

print(
    "average_candidates_per_s1:",
    counts.mean()
)

print(
    "median_candidates_per_s1:",
    np.median(counts)
)

print(
    "p95_candidates_per_s1:",
    np.quantile(counts, 0.95)
)

print(
    "p99_candidates_per_s1:",
    np.quantile(counts, 0.99)
)

print(
    "max_candidates_per_s1:",
    counts.max()
)

print(
    "macro_candidate_recall:",
    (
        combined_recall_sum / combined_evaluated
        if combined_evaluated
        else 0.0
    )
)

print(
    "n_evaluated:",
    combined_evaluated
)

print(
    "n_skipped_no_ground_truth:",
    combined_skipped
)

print(
    f"\nTime: {perf_counter() - start:.2f} seconds"
)

Processed 2,000 S1 records | candidates so far: 3,551,186
Processed 4,000 S1 records | candidates so far: 6,839,550
Processed 6,000 S1 records | candidates so far: 10,013,618
Processed 8,000 S1 records | candidates so far: 13,189,672
Processed 10,000 S1 records | candidates so far: 16,766,142

COMBINED BLOCKER — 10K TEST
------------------------------------------------------------
total_candidate_pairs: 16766142
s1_records_with_candidates: 9211
average_candidates_per_s1: 1676.6142
median_candidates_per_s1: 216.5
p95_candidates_per_s1: 8318.0
p99_candidates_per_s1: 10392.050000000001
max_candidates_per_s1: 26438
macro_candidate_recall: 0.6090772519583284
n_evaluated: 9998
n_skipped_no_ground_truth: 2

Time: 5.83 seconds


In [150]:
from collections import defaultdict

block_counts = defaultdict(int)

for i, (_, row) in enumerate(s1_val.iterrows()):

    if i >= 10_000:
        break

    name = normalize_text(row["business_name"])
    numbers = extract_address_numbers(
        row["business_address"]
    )

    # -----------------------------
    # Exact name
    # -----------------------------

    exact_name = set()

    if name:
        exact_name.update(
            s2_name_index.get(name, set())
        )
        exact_name.update(
            s3_name_index.get(name, set())
        )

    # -----------------------------
    # Rare name token
    # -----------------------------

    name_token = set()

    tokens = set(name.split()) if name else set()

    usable_tokens = [
        token
        for token in tokens
        if (
            s2_token_frequency.get(
                token,
                float("inf")
            ) <= 5000
            or
            s3_token_frequency.get(
                token,
                float("inf")
            ) <= 5000
        )
    ]

    for token in usable_tokens:
        name_token.update(
            s2_name_token_index.get(token, set())
        )
        name_token.update(
            s3_name_token_index.get(token, set())
        )

    # -----------------------------
    # Address number
    # -----------------------------

    address_number = set()

    usable_numbers = [
        number
        for number in numbers
        if (
            s2_address_number_frequency.get(
                number,
                float("inf")
            ) <= 5000
            or
            s3_address_number_frequency.get(
                number,
                float("inf")
            ) <= 5000
        )
    ]

    for number in usable_numbers:
        address_number.update(
            s2_address_number_index.get(
                number,
                set()
            )
        )
        address_number.update(
            s3_address_number_index.get(
                number,
                set()
            )
        )

    # -----------------------------
    # Name + number
    # -----------------------------

    name_number = set()

    for token in usable_tokens:
        for number in usable_numbers:

            key = (token, number)

            name_number.update(
                s2_name_number_index.get(
                    key,
                    set()
                )
            )

            name_number.update(
                s3_name_number_index.get(
                    key,
                    set()
                )
            )

    # -----------------------------
    # Track unique contribution
    # -----------------------------

    all_candidates = (
        exact_name
        | name_token
        | address_number
        | name_number
    )

    block_counts["exact_name"] += len(exact_name)
    block_counts["name_token"] += len(name_token)
    block_counts["address_number"] += len(address_number)
    block_counts["name_number"] += len(name_number)
    block_counts["combined_unique"] += len(all_candidates)

print("AVERAGE CANDIDATES PER S1")
print("-" * 50)

for key, value in block_counts.items():
    print(
        f"{key}: "
        f"{value / 10_000:.2f}"
    )

AVERAGE CANDIDATES PER S1
--------------------------------------------------
exact_name: 0.00
name_token: 0.66
address_number: 1666.80
name_number: 1.58
combined_unique: 1667.53


In [151]:
# =========================================
# RECALL CONTRIBUTION OF EACH BLOCK
# 10K VALIDATION
# =========================================

block_recall = {
    "exact_name": 0.0,
    "name_token": 0.0,
    "address_number": 0.0,
    "name_number": 0.0,
    "combined": 0.0
}

evaluated = 0
skipped = 0

for i, (_, row) in enumerate(s1_val.iterrows()):

    if i >= 10_000:
        break

    s1_id = row["entity_id"]

    if s1_id not in val_gt:
        skipped += 1
        continue

    true_matches = val_gt[s1_id]

    if len(true_matches) == 0:
        continue

    name = normalize_text(row["business_name"])

    numbers = extract_address_numbers(
        row["business_address"]
    )

    # =====================================
    # EXACT NAME
    # =====================================

    exact_name = set()

    if name:
        exact_name.update(
            s2_name_index.get(name, set())
        )
        exact_name.update(
            s3_name_index.get(name, set())
        )

    # =====================================
    # NAME TOKEN
    # =====================================

    name_token = set()

    tokens = set(name.split()) if name else set()

    usable_tokens = [
        token
        for token in tokens
        if (
            s2_token_frequency.get(
                token,
                float("inf")
            ) <= 5000
            or
            s3_token_frequency.get(
                token,
                float("inf")
            ) <= 5000
        )
    ]

    for token in usable_tokens:

        name_token.update(
            s2_name_token_index.get(
                token,
                set()
            )
        )

        name_token.update(
            s3_name_token_index.get(
                token,
                set()
            )
        )

    # =====================================
    # ADDRESS NUMBER
    # =====================================

    address_number = set()

    usable_numbers = [
        number
        for number in numbers
        if (
            s2_address_number_frequency.get(
                number,
                float("inf")
            ) <= 5000
            or
            s3_address_number_frequency.get(
                number,
                float("inf")
            ) <= 5000
        )
    ]

    for number in usable_numbers:

        address_number.update(
            s2_address_number_index.get(
                number,
                set()
            )
        )

        address_number.update(
            s3_address_number_index.get(
                number,
                set()
            )
        )

    # =====================================
    # NAME + NUMBER
    # =====================================

    name_number = set()

    for token in usable_tokens:

        for number in usable_numbers:

            key = (token, number)

            name_number.update(
                s2_name_number_index.get(
                    key,
                    set()
                )
            )

            name_number.update(
                s3_name_number_index.get(
                    key,
                    set()
                )
            )

    # =====================================
    # COMBINED UNION
    # =====================================

    combined = (
        exact_name
        | name_token
        | address_number
        | name_number
    )

    # =====================================
    # RECALL
    # =====================================

    block_recall["exact_name"] += (
        len(true_matches & exact_name)
        / len(true_matches)
    )

    block_recall["name_token"] += (
        len(true_matches & name_token)
        / len(true_matches)
    )

    block_recall["address_number"] += (
        len(true_matches & address_number)
        / len(true_matches)
    )

    block_recall["name_number"] += (
        len(true_matches & name_number)
        / len(true_matches)
    )

    block_recall["combined"] += (
        len(true_matches & combined)
        / len(true_matches)
    )

    evaluated += 1


print("\nRECALL CONTRIBUTION — 10K TEST")
print("-" * 60)

for block, value in block_recall.items():

    print(
        f"{block}: "
        f"{value / evaluated:.6f}"
    )

print("\nEvaluated:", evaluated)
print("Skipped:", skipped)


RECALL CONTRIBUTION — 10K TEST
------------------------------------------------------------
exact_name: 0.000000
name_token: 0.000141
address_number: 0.464222
name_number: 0.310250
combined: 0.475366

Evaluated: 9425
Skipped: 2


In [152]:
[name for name in globals()
 if "name" in name.lower()
 and "index" in name.lower()]

['build_name_index',
 's2_name_index',
 's3_name_index',
 's2_name_token_index',
 's3_name_token_index',
 'build_full_name_index',
 's2_full_name_index',
 's3_full_name_index',
 'build_name_number_index',
 's2_name_number_index',
 's3_name_number_index']

In [153]:
# Compare the two name indexes on the same 10K validation records

exact_name_recall = {
    "name_index": 0.0,
    "full_name_index": 0.0
}

evaluated = 0

for i, (_, row) in enumerate(s1_val.iterrows()):

    if i >= 10_000:
        break

    s1_id = row["entity_id"]

    if s1_id not in val_gt:
        continue

    true_matches = val_gt[s1_id]

    if len(true_matches) == 0:
        continue

    name = normalize_text(row["business_name"])

    # -----------------------------------------
    # s2_name_index / s3_name_index
    # -----------------------------------------

    candidates_name = set()

    if name:
        candidates_name.update(
            s2_name_index.get(name, set())
        )
        candidates_name.update(
            s3_name_index.get(name, set())
        )

    # -----------------------------------------
    # s2_full_name_index / s3_full_name_index
    # -----------------------------------------

    candidates_full_name = set()

    if name:
        candidates_full_name.update(
            s2_full_name_index.get(name, set())
        )
        candidates_full_name.update(
            s3_full_name_index.get(name, set())
        )

    # -----------------------------------------
    # Recall
    # -----------------------------------------

    exact_name_recall["name_index"] += (
        len(true_matches & candidates_name)
        / len(true_matches)
    )

    exact_name_recall["full_name_index"] += (
        len(true_matches & candidates_full_name)
        / len(true_matches)
    )

    evaluated += 1


print("EXACT NAME INDEX COMPARISON")
print("-" * 50)

for key, value in exact_name_recall.items():

    print(
        f"{key}: "
        f"{value / evaluated:.6f}"
    )

print("\nEvaluated:", evaluated)

EXACT NAME INDEX COMPARISON
--------------------------------------------------
name_index: 0.000000
full_name_index: 0.221225

Evaluated: 9425
